# Option A – v3: protocol census and ZINB baselines

This last run **adds to** v2 (`optionA_outputs_v2.zip`, which already holds v1). It answers the two questions a reviewer is most likely to ask:

| Part | What runs | Question |
|---|---|---|
| **VII. Protocol census** | 7 public crime-prediction repositories (2021–2026: ST-SHN, ST-HSL, HAGEN, CL4ST, STExplainer, EasyST, ST-HHOL), cloned at pinned commits, audited automatically | Is scoring only the non-zero cells an old habit, or still standard practice in new models? |
| **III / IV / I.5. ZINB baselines** | Zero-inflated negative-binomial intervals: fitted on the calibration block (global and per volume bin) and a learned GRU-ZINB; main split (H = 1–4, 3 seeds), rolling years 2020–2024, daily benchmark | Does a distributional model built for excess zeros already give calibrated intervals, making conformal calibration unnecessary? |
| **Check** | The weekly panels are rebuilt (cut at v1's last week) and v1's interval numbers recomputed | Are the v3 numbers comparable with v1? |

Recent public crime models could not be re-run on the benchmark: ST-HHOL's data are not released, and CL4ST / STExplainer / EasyST ship no pipeline for the NYC/CHI benchmark (EasyST's crime data are a different Chicago grid). The census records this.

**Before you run (one time)**
1. Make a Kaggle dataset from **`optionA_outputs_v2.zip`** and add it as input (*Add Input*). The notebook finds it anywhere under `/kaggle/input`.
2. *Settings → Accelerator → GPU T4 x2*, *Internet → On* (no Census key needed this time).
3. **Save Version → Save & Run All (Commit).** Expected run time 1–2 hours.

**What to send back:** `optionA_outputs_v3.zip` (the complete v1 + v2 + v3 result set).

## Setup (as in v1, output zip `optionA_outputs_v3.zip`) and the v2 result set copied in

In [ ]:
import os, sys, time, json, math, gc, re, warnings, zipfile, shutil, subprocess, random, traceback, hashlib, io, contextlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
T0 = time.time()

# ------------------------------------------------------------------ configuration
FAST        = os.environ.get("OA_FAST", "0") == "1"          # quick smoke test only
TIME_LIMIT_H = 11.4                                           # stop optional work before Kaggle's 12-hour limit
START_YEAR  = int(os.environ.get("OA_START_YEAR", "2016"))
TRAIN_END   = pd.Timestamp("2023-12-31")                      # same split as the capstone
VAL_END     = pd.Timestamp("2024-06-30")                      # calibration / validation block; test = after this date
CITIES      = ["CHI", "NYC", "SF"]
CITY_NAME   = {"CHI": "Chicago", "NYC": "New York", "SF": "San Francisco"}
DROP_LAST_WEEKS = {"CHI": 1, "NYC": 4, "SF": 2}               # late-reported records make the newest weeks incomplete
HORIZONS    = [1, 2, 3, 4]
NN_HORIZONS = [1, 4]                                          # neural forecasters (slower) are run for these horizons
SEEDS_TREE  = [0] if FAST else [0, 1, 2]
SEEDS_NN    = [0] if FAST else [0, 1, 2]
SEEDS_BENCH = [0] if FAST else [0, 1, 2]                      # ST-HSL, Graph WaveNet, MTGNN, our benchmark forecaster
SEEDS_STSHN = [0] if FAST else [0, 1]                         # ST-SHN (TensorFlow, batch size 1, slower)
ROLL_YEARS  = [2021, 2022] if FAST else [2020, 2021, 2022, 2023, 2024]
ALPHA       = 0.10                                            # 90 % intervals
SEED        = 42

KAGGLE = Path("/kaggle/working").exists()
WORK   = Path("/kaggle/working") if KAGGLE else Path(os.environ.get("OA_WORK", Path.cwd()))
OUT    = WORK / "optionA_outputs"; FIG = OUT / "figures"; TAB = OUT / "tables"; LOGS = OUT / "logs"; PRED = OUT / "predictions"
DATA   = WORK / "raw_data"; REPOS = WORK / "repos"
for p in (OUT, FIG, TAB, LOGS, PRED, DATA, REPOS): p.mkdir(parents=True, exist_ok=True)

def hours_used(): return (time.time() - T0) / 3600
def time_left(): return TIME_LIMIT_H - hours_used()
def log(msg):
    line = f"[{hours_used():5.2f} h] {msg}"; print(line, flush=True)
    with open(LOGS / "progress.log", "a") as f: f.write(line + "\n")
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=140, bbox_inches="tight"); plt.show(); plt.close("all")
def save_table(df, name, index=False):
    df.to_csv(TAB / f"{name}.csv", index=index); return df
def refresh_zip():
    """Zip everything in the output folder (called after every part)."""
    dest = WORK / "optionA_outputs_v3.zip"; tmp = WORK / "optionA_outputs_v3.zip.part"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob("*"):
            if p.is_file(): z.write(p, "optionA_outputs/" + str(p.relative_to(OUT)))
    tmp.replace(dest); log(f"zip refreshed: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

STATUS = {}
@contextlib.contextmanager
def part(name):
    """Run one part; on an error write the traceback to logs/errors.log and continue with the next part."""
    t = time.time(); log(f"===== {name} started"); STATUS[name] = "running"
    try:
        yield; STATUS[name] = f"ok ({(time.time() - t) / 60:.0f} min)"
    except Exception:
        tb = traceback.format_exc(); STATUS[name] = "FAILED – see logs/errors.log"
        print(tb); open(LOGS / "errors.log", "a").write(f"===== {name}\n{tb}\n")
    finally:
        log(f"===== {name}: {STATUS[name]}"); json.dump(STATUS, open(LOGS / "status.json", "w"), indent=1)
        try: refresh_zip()
        except Exception as e: print("zip failed:", e)

def pip_install(*pkgs):
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], capture_output=True, timeout=900)
try: import lightgbm as lgb
except ImportError: pip_install("lightgbm"); import lightgbm as lgb
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy.stats import poisson as _pois, nbinom as _nb, kendalltau, spearmanr

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()

def pick_device():
    if torch.cuda.is_available():
        try: torch.ones(8, device="cuda").sum().item(); return torch.device("cuda")
        except Exception as e: print("CUDA present but unusable with this PyTorch build:", str(e)[:120])
    return torch.device("cpu")
DEVICE = pick_device()
def xgb_device():
    try:
        b = xgb.XGBRegressor(n_estimators=2, device="cuda", tree_method="hist"); b.fit(np.random.rand(64, 3), np.random.rand(64)); return "cuda"
    except Exception: return "cpu"
XGB_DEVICE = xgb_device() if DEVICE.type == "cuda" else "cpu"
VERSIONS = dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, torch=torch.__version__, xgboost=xgb.__version__,
                lightgbm=lgb.__version__, device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none", fast=FAST)
json.dump(VERSIONS, open(LOGS / "environment.json", "w"), indent=1)
log(f"environment: {VERSIONS}")
if DEVICE.type == "cpu" and not FAST: print("WARNING: no usable GPU. Switch the accelerator to 'GPU T4 x2', or expect a much longer run.")

# ------------------------------------------------------------------ shared metric helpers
def pdev(y, mu):
    mu = np.maximum(np.asarray(mu, float), 1e-6); y = np.asarray(y, float)
    return 2 * np.mean(np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0) - (y - mu))
def unit_dev(y, mu):
    mu = np.maximum(np.asarray(mu, float), 1e-6); y = np.asarray(y, float)
    return 2 * (np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0) - (y - mu))
def block_boot(diff, blocks, B=1000, seed=SEED):
    """Mean of `diff` and a 95 % CI from a bootstrap that resamples whole blocks (weeks or days)."""
    wk = pd.factorize(np.asarray(blocks))[0]; s = np.bincount(wk, diff); n = np.bincount(wk); r = np.random.default_rng(seed)
    idx = r.integers(0, len(s), (B, len(s))); est = s[idx].sum(1) / n[idx].sum(1)
    return float(np.mean(diff)), float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

# ------------------------------------------------------------------ shared forecasters (gradient-boosted Poisson models)
def fit_lgb_pois(Xtr, ytr, Xes, yes, seed=0, n_est=None):
    m = lgb.LGBMRegressor(objective="poisson", n_estimators=n_est or (150 if FAST else 3000), learning_rate=0.05, num_leaves=127, min_child_samples=20,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m
def fit_xgb_pois(Xtr, ytr, Xes, yes, seed=0):
    m = xgb.XGBRegressor(objective="count:poisson", n_estimators=150 if FAST else 2000, learning_rate=0.05, max_depth=7, subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=5, tree_method="hist", device=XGB_DEVICE, enable_categorical=True,
                         max_cat_to_onehot=1, random_state=seed, early_stopping_rounds=50, eval_metric="poisson-nloglik")
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], verbose=False); return m
def fit_lgb_quantile(Xtr, ytr, Xes, yes, q, seed=0):
    m = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=150 if FAST else 2000, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m
def fit_lgb_l2(Xtr, ytr, Xes, yes, seed=0):
    m = lgb.LGBMRegressor(objective="l2", n_estimators=150 if FAST else 2000, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m

# ------------------------------------------------------------------ shared interval helpers (all calibrated on a held-out block only)
def conformal_q(scores, alpha=ALPHA):
    scores = np.asarray(scores, float); n = len(scores)
    if n == 0: return np.inf
    lev = np.ceil((n + 1) * (1 - alpha)) / n
    return np.inf if lev > 1 else float(np.quantile(scores, lev, method="higher"))
def norm_score(y, mu): return np.abs(y - mu) / np.sqrt(mu + 0.5)
def int_from_q(mu, q, scale=None):
    s = np.sqrt(mu + 0.5) if scale is None else scale
    lo = np.maximum(np.ceil(mu - q * s), 0); hi = np.floor(mu + q * s)
    bad = hi < lo; r = np.round(mu); return np.where(bad, r, lo), np.where(bad, r, hi)
def mondrian_edges(mu_cal, k=5): return np.unique(np.quantile(mu_cal, np.linspace(0, 1, k + 1)[1:-1]))
def nb_size(mu, y):                                            # negative-binomial size by moments: Var = mu + mu^2 / r
    return float(min(max(np.sum(mu ** 2) / max(np.sum((y - mu) ** 2 - mu), 1e-6), 1e-3), 1e4))
def parametric_interval(mu, kind, r=None):
    mu = np.maximum(mu, 1e-6)
    if kind == "Poisson": return _pois.ppf(ALPHA / 2, mu), _pois.ppf(1 - ALPHA / 2, mu)
    p = r / (r + mu); return _nb.ppf(ALPHA / 2, r, p), _nb.ppf(1 - ALPHA / 2, r, p)
def interval_score(y, lo, hi): return (hi - lo) + (2 / ALPHA) * np.maximum(lo - y, 0) + (2 / ALPHA) * np.maximum(y - hi, 0)
def coverage_report(y, lo, hi, vq=None, area=None):
    """Overall coverage, width, interval score, coverage per volume quintile and per area."""
    cov = (y >= lo) & (y <= hi); out = dict(coverage=cov.mean(), width=(hi - lo).mean(), interval_score=interval_score(y, lo, hi).mean())
    if vq is not None:
        for q in range(1, 6): out[f"cov_Q{q}"] = cov[vq == q].mean() if (vq == q).any() else np.nan
        out["max_abs_dev_from_90"] = np.nanmax([abs(out[f"cov_Q{q}"] - (1 - ALPHA)) for q in range(1, 6)])
        out["gap_Q5_minus_Q1"] = out["cov_Q5"] - out["cov_Q1"]
    if area is not None:
        ac = pd.Series(cov).groupby(np.asarray(area)).mean()
        out["areas_within_5pts"] = float((ac.sub(1 - ALPHA).abs() <= 0.05).mean()); out["worst_area_dev"] = float(ac.sub(1 - ALPHA).abs().max())
        out["sd_area_coverage"] = float(ac.std())
    return out

In [ ]:
# ------------------------------------------------------------------ v3 configuration: start from the complete v2 result set
INPUT_ROOT = Path(os.environ.get("OA_INPUT", "/kaggle/input"))
def find_prev():
    """Folder of the v2 outputs (the only version that has tables/b09b_reproduction_gap_ci.csv)."""
    hits = sorted(INPUT_ROOT.rglob("b09b_reproduction_gap_ci.csv")) if INPUT_ROOT.exists() else []
    hits = [h for h in hits if not (h.parent.parent / "logs_v2").exists()]      # skip the output of an earlier v3 session
    if not hits: raise FileNotFoundError("v2 outputs not found under /kaggle/input: add the dataset made from optionA_outputs_v2.zip (Add Input).")
    return hits[0].parent.parent
PREV = find_prev(); log(f"v2 outputs: {PREV}")
for sub in ("tables", "figures", "predictions", "census", "logs_v1"):
    if (PREV / sub).exists(): shutil.copytree(PREV / sub, OUT / sub, dirs_exist_ok=True)
if (PREV / "logs").exists(): shutil.copytree(PREV / "logs", OUT / "logs_v2", dirs_exist_ok=True)
for f in PREV.glob("*.md"): shutil.copy(f, OUT / (f.stem + ("_v2" if f.stem == "results_summary" else "") + f.suffix))
for f in PREV.glob("*.json"): shutil.copy(f, OUT / (f.stem + ("_v2" if f.stem == "metrics_summary" else "") + f.suffix))
STATUS_PREV = json.load(open(PREV / "logs" / "status.json")) if (PREV / "logs" / "status.json").exists() else {}
def tab_prev(name, **kw):
    p = PREV / "tables" / f"{name}.csv"
    return pd.read_csv(p, **kw) if p.exists() and p.stat().st_size > 2 else None
# the weekly panels are cut at v1's last week, so that v3 tests on exactly the weeks of v1
_ps = tab_prev("c02_panel_summary")
V1_LAST_WEEK = {r.city: pd.Timestamp(r.last_week) for r in _ps.itertuples()} if _ps is not None else {}
V1_PANEL = _ps
log(f"v1 last weeks: { {k: str(v.date()) for k, v in V1_LAST_WEEK.items()} }")

## Part I.1 Benchmark repositories and audit (as in v1; needed for the benchmark data)

In [ ]:
BENCH_CATS = {"NYC": ["Burglary", "Larceny", "Robbery", "Assault"], "CHI": ["Theft", "Battery", "Assault", "Damage"]}
BENCH_CITIES = ["NYC", "CHI"]
# Table III of Li et al. (2022, ICDE), ST-HSL paper: MAE, MAPE per category. Verify against the paper before quoting.
PUBLISHED = {
    "NYC": {"ARIMA": [0.8999, 0.6305, 1.3015, 0.6268, 0.9558, 0.5969, 0.9983, 0.6198], "SVM": [1.1604, 0.7653, 1.4979, 0.6417, 1.1278, 0.6733, 1.1928, 0.6964],
            "ST-ResNet": [0.8680, 0.5603, 1.1082, 0.5329, 0.8717, 0.5209, 0.9645, 0.5749], "DCRNN": [0.8176, 0.5324, 1.0732, 0.5492, 0.9189, 0.5532, 0.9692, 0.5955],
            "STGCN": [0.8366, 0.5404, 1.0629, 0.5295, 0.9035, 0.5441, 0.9375, 0.5757], "GWN": [0.7993, 0.5235, 1.0493, 0.5405, 0.8681, 0.5351, 0.8866, 0.5646],
            "STtrans": [0.8617, 0.5592, 1.0896, 0.5478, 0.8839, 0.5651, 0.9363, 0.5679], "DeepCrime": [0.8227, 0.5508, 1.0618, 0.5351, 0.8841, 0.5537, 0.9222, 0.5677],
            "MTGNN": [0.8329, 0.5439, 1.0473, 0.5330, 0.8759, 0.5457, 0.9090, 0.5714], "ST-SHN": [0.8012, 0.5198, 1.0431, 0.5291, 0.8717, 0.5362, 0.9169, 0.5682],
            "DMSTGCN": [0.8376, 0.5485, 1.0410, 0.5464, 0.8597, 0.5403, 0.9036, 0.5601], "ST-HSL": [0.7329, 0.4788, 1.0316, 0.5040, 0.7912, 0.4595, 0.8484, 0.5029]},
    "CHI": {"ARIMA": [1.5965, 0.5720, 1.3212, 0.5792, 0.8691, 0.6044, 1.0430, 0.6134], "SVM": [1.7711, 0.5629, 1.3493, 0.6027, 1.0879, 0.6560, 1.1313, 0.5721],
            "ST-ResNet": [1.3931, 0.5488, 1.1519, 0.5719, 0.7679, 0.4633, 0.9064, 0.5018], "DCRNN": [1.3699, 0.5770, 1.1583, 0.5528, 0.7639, 0.4600, 0.8764, 0.4756],
            "STGCN": [1.3628, 0.5359, 1.1512, 0.5761, 0.7963, 0.4810, 0.9068, 0.4959], "GWN": [1.3211, 0.5502, 1.1331, 0.5503, 0.7493, 0.4580, 0.8584, 0.4850],
            "STtrans": [1.3404, 0.5356, 1.1466, 0.5684, 0.7671, 0.4499, 0.8987, 0.4842], "DeepCrime": [1.3391, 0.5430, 1.1290, 0.5389, 0.7737, 0.4616, 0.9096, 0.4960],
            "ST-MetaNet": [1.3369, 0.5369, 1.1762, 0.5748, 0.7904, 0.4753, 0.8907, 0.4756], "MTGNN": [1.3054, 0.5378, 1.1307, 0.5597, 0.7571, 0.4572, 0.8667, 0.4859],
            "ST-SHN": [1.3231, 0.5310, 1.1348, 0.5544, 0.7758, 0.4574, 0.8741, 0.4747], "DMSTGCN": [1.3292, 0.5291, 1.1297, 0.5552, 0.8058, 0.4759, 0.8698, 0.4877],
            "ST-HSL": [1.2952, 0.4929, 1.1016, 0.5231, 0.6665, 0.3996, 0.8446, 0.4644]}}
REPO_URLS = {"STHSL": "https://github.com/LZH-YS1998/STHSL", "ST-SHN": "https://github.com/akaxlh/ST-SHN",
             "Graph-WaveNet": "https://github.com/nnzhan/Graph-WaveNet", "MTGNN": "https://github.com/nnzhan/MTGNN"}
PATCHES = {"STHSL": "engine.py: running loss stored with .item() (memory only); train.py: device and seed read from arguments/environment (official default seed 523 = our seed 0)",
           "ST-SHN": "TensorFlow 1 code run with tf.compat.v1; xavier_initializer → glorot_uniform (same initialiser); softmax on 6-D tensor written out (numerically identical); seed from environment; predictions and validation metrics logged every epoch",
           "Graph-WaveNet": "nn.Conv1d with 2-D kernels replaced by nn.Conv2d (old PyTorch accepted this silently; identical computation)",
           "MTGNN": "none"}

def git_clone(name):
    dest = REPOS / name
    if not dest.exists():
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URLS[name], str(dest)], capture_output=True, text=True)
        if r.returncode != 0: raise RuntimeError(f"could not clone {name}: {r.stderr[-300:]}")
    h = subprocess.run(["git", "rev-parse", "HEAD"], cwd=dest, capture_output=True, text=True).stdout.strip()
    return dest, h

def md5(p): return hashlib.md5(open(p, "rb").read()).hexdigest()
def bench_load(c):
    import pickle
    t = {s: np.asarray(pickle.load(open(REPOS / "STHSL" / "Datasets" / f"{c}_crime" / f"{s}.pkl", "rb")), np.float32) for s in ("trn", "val", "tst")}
    return np.concatenate([t["trn"], t["val"], t["tst"]], 2), t["trn"].shape[2], t["val"].shape[2], t["tst"].shape[2]

from pathlib import Path
def patch_stshn(root):
    """Minimal patches so the official TF1 ST-SHN code runs on TF2 (compat.v1), plus logging hooks.
    Model, loss, data handling and the official metrics are unchanged."""
    root = Path(root)
    nn_ = root / "Utils" / "NNLayers.py"; s = nn_.read_text()
    if "compat.v1" not in s:
        s = s.replace("import tensorflow as tf\nfrom tensorflow.contrib.layers import xavier_initializer",
                      "import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()\nxavier_initializer = tf.glorot_uniform_initializer", 1)
        assert "compat.v1" in s, "NNLayers import pattern not found"; nn_.write_text(s)
    m = root / "HG_ST_labcode.py"; s = m.read_text()
    if "compat.v1" in s: return
    rep = [
        ("import tensorflow as tf\n", "import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()\nimport os as _os, json as _json\n"
         "def _softmax(x, axis):\n\tx = x - tf.reduce_max(x, axis=axis, keepdims=True); e = tf.exp(x)\n\treturn e / tf.reduce_sum(e, axis=axis, keepdims=True)\n"),
        # hook 1: keep the per-cell predictions of every evaluation pass (aligned with the day index)
        ("\t\tnum = len(ids)\n\n\t\tsteps = int(np.ceil(num / args.batch))\n\t\tfor i in range(steps):\n\t\t\tst = i * args.batch\n\t\t\ted = min((i+1) * args.batch, num)\n\t\t\tbatIds = ids[st: ed]\n\n\t\t\ttem = self.sampTestBatch(batIds, tstTensor, inpTensor)",
         "\t\tnum = len(ids)\n\t\tself._P = np.zeros([args.areaNum, tstTensor.shape[1], args.offNum], np.float32)\n\n\t\tsteps = int(np.ceil(num / args.batch))\n\t\tfor i in range(steps):\n\t\t\tst = i * args.batch\n\t\t\ted = min((i+1) * args.batch, num)\n\t\t\tbatIds = ids[st: ed]\n\n\t\t\ttem = self.sampTestBatch(batIds, tstTensor, inpTensor)"),
        ("\t\t\t\tpreds, preLoss, regLoss, loss, sqLoss, absLoss, tstNums, apeLoss, posNums = res\n",
         "\t\t\t\tpreds, preLoss, regLoss, loss, sqLoss, absLoss, tstNums, apeLoss, posNums = res\n\t\t\t\tself._P[:, batIds[0], :] = preds\n"),
        # hook 2: after every epoch evaluate the validation block too, and store validation metrics + test predictions
        ("\t\t\tif test:\n\t\t\t\treses = self.testEpoch(self.handler.tstT, np.concatenate([self.handler.trnT, self.handler.valT], axis=1))\n",
         "\t\t\tif test:\n\t\t\t\tvres = self.testEpoch(self.handler.valT, self.handler.trnT)\n\t\t\t\treses = self.testEpoch(self.handler.tstT, np.concatenate([self.handler.trnT, self.handler.valT], axis=1))\n\t\t\t\tself._hist.append(dict(epoch=ep, val=vres, test=reses)); self._preds.append(self._P.copy())\n\t\t\t\tnp.save(_os.environ.get('STSHN_OUT', 'stshn') + '_preds.npy', np.stack(self._preds)); _json.dump([{k: {kk: float(vv) for kk, vv in v.items()} if isinstance(v, dict) else v for k, v in h.items()} for h in self._hist], open(_os.environ.get('STSHN_OUT', 'stshn') + '_hist.json', 'w'))\n"),
        ("\t\tbestRes = None\n\t\tfor ep in range(stloc, args.epoch):", "\t\tbestRes = None\n\t\tself._hist, self._preds = [], []\n\t\tfor ep in range(stloc, args.epoch):"),
        # seed from the environment (the official code sets none)
        ("\tconfig = tf.ConfigProto()\n", "\t_sd = int(_os.environ.get('STSHN_SEED', '0')); np.random.seed(_sd); tf.set_random_seed(_sd)\n\tconfig = tf.ConfigProto()\n"),
    ]
    for a, b in rep:
        assert a in s, "ST-SHN patch pattern not found: " + a[:60]
        s = s.replace(a, b, 1)
    assert s.count("att = tf.nn.softmax(") == 2, "ST-SHN softmax pattern not found"
    s = s.replace("att = tf.nn.softmax(", "att = _softmax(")    # TF2 kernels reject softmax on 6-D tensors; same softmax written out
    m.write_text(s)

with part("I.1 benchmark: code, data and protocol audit"):
    VERS = []
    for n in REPO_URLS:
        try: d, h = git_clone(n); VERS.append(dict(repository=n, url=REPO_URLS[n], commit=h, patches=PATCHES[n]))
        except Exception as e: VERS.append(dict(repository=n, url=REPO_URLS[n], commit="NOT AVAILABLE", patches=str(e)[:200]))
    VERS = save_table(pd.DataFrame(VERS), "b01_code_versions_and_patches"); display(VERS)
    BENCH_OK = (REPOS / "STHSL" / "Datasets").exists()
    # same data in both official repositories?
    same = []
    for c in BENCH_CITIES:
        for s in ("trn", "val", "tst"):
            a, b = REPOS / "STHSL" / "Datasets" / f"{c}_crime" / f"{s}.pkl", REPOS / "ST-SHN" / "Datasets" / f"{c}_crime" / f"{s}.pkl"
            same.append(dict(city=c, split=s, sthsl_md5=md5(a) if a.exists() else None, stshn_md5=md5(b) if b.exists() else None))
    same = pd.DataFrame(same); same["identical"] = same["sthsl_md5"] == same["stshn_md5"]; save_table(same, "b02_dataset_identity"); display(same)

    # automatic audit of the evaluation code (run on the untouched clones, before any patch)
    def code_has(repo, file, pattern):
        p = REPOS / repo / file
        return bool(p.exists() and re.search(pattern, p.read_text()))
    AUDIT = [
        ("ST-HSL", "scored cell-days", "only cell-days with at least one crime (mask = label > 0)", code_has("STHSL", "engine.py", r"mask = 1 \* \(label > 0\)")),
        ("ST-HSL", "training loss", "squared error on every cell-day (mask = label >= 0)", code_has("STHSL", "engine.py", r"retLabels = \(label >= 0\) \* 1")),
        ("ST-HSL", "reported epoch", "best validation RMSE + MAE", code_has("STHSL", "train.py", r"val_metrics = res_eval\['RMSE'\] \+ res_eval\['MAE'\]")),
        ("ST-HSL", "random seed", "fixed (seed_torch, 523)", code_has("STHSL", "utils.py", r"def seed_torch\(seed=523\)")),
        ("ST-SHN", "scored cell-days", "crime days only, and only cells with crime on 20–80 % of training days",
         code_has("ST-SHN", "HG_ST_labcode.py", r"mask = self\.handler\.tstLocs \* \(label > 0\)") and code_has("ST-SHN", "DataHandler.py", r"\(percent > 0\.2\) \* \(percent < 0\.8\)")),
        ("ST-SHN", "training loss", "squared error on crime days only (regression: negRate = 0)", code_has("ST-SHN", "HG_ST_labcode.py", r"negRate = 0")),
        ("ST-SHN", "reported epoch", "epoch with the best TEST MAPE (the test set chooses the epoch)",
         code_has("ST-SHN", "HG_ST_labcode.py", r"reses = self\.testEpoch\(self\.handler\.tstT.*\n.*bestRes\['MAPE'\] > reses\['MAPE'\]")),
        ("ST-SHN", "random seed", "none set", not code_has("ST-SHN", "HG_ST_labcode.py", r"set_random_seed|np\.random\.seed")),
    ]
    AUDIT = save_table(pd.DataFrame(AUDIT, columns=["code", "aspect", "what the official code does", "verified in code"]), "b03_protocol_audit"); display(AUDIT)

    # how much of the test set does the official metric ignore?
    BSTAT = []
    for c in BENCH_CITIES:
        Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; tst = Xall[:, :, ntr + nva:].reshape(R * Cc, nte, K)
        act = (Xall[:, :, :ntr] > 0).mean(2).reshape(R * Cc, K); tl = (act > 0.2) & (act < 0.8)
        for k, n in enumerate(BENCH_CATS[c]):
            y = tst[:, :, k]
            BSTAT.append(dict(city=c, category=n, cells=R * Cc, test_days=nte, mean_daily_count=y.mean(), share_zero_cell_days=(y == 0).mean(),
                              share_scored_P1=(y > 0).mean(), cells_in_P2_subset=int(tl[:, k].sum()), share_scored_P2=((y > 0) & tl[:, k][:, None]).mean()))
    BSTAT = save_table(pd.DataFrame(BSTAT), "b04_benchmark_sparsity"); display(BSTAT.round(3))

In [ ]:
def bench_features(X, days):
    """X: (rows, cols, days, K). One row per (day, cell, category) in the order used by the official code."""
    R, Cc, T, K = X.shape; d = np.asarray(days)
    cs = np.concatenate([np.zeros((R, Cc, 1, K), np.float32), np.cumsum(X, 2)], 2)
    Pd = np.pad(X, ((1, 1), (1, 1), (0, 0), (0, 0))); NB = sum(Pd[1 + dy:1 + dy + R, 1 + dx:1 + dx + Cc] for dy in (-1, 0, 1) for dx in (-1, 0, 1)) - X
    csn = np.concatenate([np.zeros((R, Cc, 1, K), np.float32), np.cumsum(NB, 2)], 2)
    tot = X.sum(3, keepdims=True); cst = np.concatenate([np.zeros((R, Cc, 1, 1), np.float32), np.cumsum(tot, 2)], 2)
    f = {f"lag{l}": X[:, :, d - l] for l in range(1, 8)}
    for w in (7, 14, 30): f[f"m{w}"] = (cs[:, :, d] - cs[:, :, d - w]) / w
    f["same_dow4"] = np.mean([X[:, :, d - 7 * j] for j in (1, 2, 3, 4)], 0)
    f["nb1"] = NB[:, :, d - 1]; f["nb7"] = (csn[:, :, d] - csn[:, :, d - 7]) / 7; f["nb30"] = (csn[:, :, d] - csn[:, :, d - 30]) / 30
    f["cell_all7"] = np.broadcast_to((cst[:, :, d] - cst[:, :, d - 7]) / 7, f["lag1"].shape)
    f["city_lag1"] = np.broadcast_to(X.sum((0, 1), keepdims=True)[:, :, d - 1], f["lag1"].shape)
    df = pd.DataFrame({k: np.ascontiguousarray(np.moveaxis(v, 2, 0)).reshape(-1) for k, v in f.items()})
    n = len(d); df["cell"] = pd.Categorical(np.tile(np.repeat(np.arange(R * Cc), K), n)); df["cat"] = pd.Categorical(np.tile(np.arange(K), n * R * Cc))
    df["dow"] = np.repeat(d % 7, R * Cc * K)
    return df, np.moveaxis(X[:, :, d], 2, 0).reshape(-1)
def ztp_median(mu):
    mu = np.maximum(mu, 1e-3); p0 = np.exp(-mu); out = np.ones_like(mu)
    for k in range(1, 80): out = np.where((_pois.cdf(k, mu) - p0) / (1 - p0) < 0.5, k + 1, out)
    return out

# ---------------------------------------------------------------- ST-HSL (official code)
def prep_sthsl():
    d = REPOS / "STHSL"; eg = (d / "engine.py").read_text()
    if "epochLoss += loss\n" in eg: (d / "engine.py").write_text(eg.replace("epochLoss += loss\n", "epochLoss += loss.item()\n", 1))
    tr = (d / "train.py").read_text()
    if "torch.device(args.device)" not in tr: tr = tr.replace("torch.device('cuda:0')", "torch.device(args.device)")
    if "STHSL_SEED" not in tr:
        assert "    seed_torch()\n" in tr, "ST-HSL seed pattern not found"
        tr = tr.replace("    seed_torch()\n", "    import os as _os; seed_torch(int(_os.environ.get('STHSL_SEED', '523')))\n", 1)
    (d / "train.py").write_text(tr)
    (d / "dump_preds.py").write_text('''
import numpy as np, torch
from Params import args
from DataHandler import DataHandler
from model import STHSL
import engine
handler = DataHandler(); model = STHSL().to(args.device)
model.load_state_dict(torch.load(args.checkpoint, map_location=args.device)); model.eval()
def run(T, inp):
    ids = np.arange(T.shape[1]); P = []
    with torch.no_grad():
        for st in range(0, len(ids), args.batch):
            b = ids[st:st + args.batch]
            feats, labels, mask = engine.sampleTestBatch(b, st, st + len(b), T, inp, handler)
            feats = torch.Tensor(feats).to(args.device)
            out = model(feats, feats[:, np.random.permutation(args.areaNum)])[-1]
            P.append(handler.zInverse(out).cpu().numpy())
    return np.concatenate(P)
np.save("sthsl_pred_tst.npy", run(handler.tstT, np.concatenate([handler.trnT, handler.valT], axis=1)))
np.save("sthsl_pred_val.npy", run(handler.valT, handler.trnT))
''')

def run_logged(cmd, cwd, logfile, timeout_s, env=None):
    with open(logfile, "w") as lf:
        p = subprocess.Popen(cmd, cwd=cwd, stdout=lf, stderr=subprocess.STDOUT, env={**os.environ, **(env or {})})
        try: rc = p.wait(timeout=timeout_s)
        except subprocess.TimeoutExpired: p.kill(); rc = "timeout"
    return rc

def run_sthsl(c, seed, epochs, timeout_s):
    d = REPOS / "STHSL"; (d / "Save" / c).mkdir(parents=True, exist_ok=True)
    for f in (d / "Save" / c).glob("*.pth"): f.unlink()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    dev = "cuda:0" if DEVICE.type == "cuda" else "cpu"; t = time.time(); lf = LOGS / f"sthsl_{c}_seed{seed}.log"
    rc = run_logged([sys.executable, "train.py", "--data", c, "--device", dev, "--epoch", str(epochs), "--save", "./Save/"], d, lf, timeout_s,
                    env={"STHSL_SEED": str(523 if seed == 0 else seed)})
    txt = lf.read_text(errors="ignore").replace("\r", "\n"); ep, best_ep, n_ep = 0, None, 0
    for line in txt.splitlines():
        mm = re.match(r"Epoch\s+(\d+) Training Time", line.strip())
        if mm: ep = int(mm.group(1)); n_ep = ep
        if "Val metrics decrease" in line: best_ep = ep
    if best_ep is None: log(f"  ST-HSL {c} seed {seed}: no finished epoch (exit {rc}); see {lf.name}"); return None
    ck = sorted((d / "Save" / c).glob(f"_epoch_{best_ep}_MAE_*.pth"))
    if not ck: log(f"  ST-HSL {c} seed {seed}: checkpoint of epoch {best_ep} missing"); return None
    rc2 = run_logged([sys.executable, "dump_preds.py", "--data", c, "--device", dev, "--checkpoint", str(ck[0])], d, LOGS / f"sthsl_{c}_seed{seed}_dump.log", 1800)
    if rc2 != 0: log(f"  ST-HSL {c} seed {seed}: prediction dump failed ({rc2})"); return None
    info = dict(model="ST-HSL", city=c, seed=seed, epochs_run=n_ep, epochs_planned=epochs, selected_epoch=best_ep, selection="validation (official rule)",
                minutes=(time.time() - t) / 60, finished=(rc == 0))
    log(f"  ST-HSL {c} seed {seed}: {n_ep}/{epochs} epochs, validation-best epoch {best_ep}, {info['minutes']:.0f} min")
    # (days, cells, K) predictions
    return np.load(d / "sthsl_pred_tst.npy"), np.load(d / "sthsl_pred_val.npy"), info

# ---------------------------------------------------------------- ST-SHN (official TensorFlow 1 code)
def prep_stshn():
    src, dst = REPOS / "ST-SHN", REPOS / "ST-SHN-run"
    if not dst.exists(): shutil.copytree(src, dst)
    patch_stshn(dst); (dst / "History").mkdir(exist_ok=True); (dst / "Models").mkdir(exist_ok=True)
    return dst
def run_stshn(c, seed, timeout_s, epochs=None):
    d = REPOS / "ST-SHN-run"; t = time.time(); lf = LOGS / f"stshn_{c}_seed{seed}.log"; outp = str(d / f"out_{c}_s{seed}")
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    cmd = [sys.executable, "HG_ST_labcode.py", "--data", c, "--task", "r", "--save_path", f"{c}_s{seed}"] + (["--epoch", str(epochs)] if epochs else [])
    env = {"STSHN_OUT": outp, "STSHN_SEED": str(seed), "TF_CPP_MIN_LOG_LEVEL": "2"}
    if DEVICE.type != "cuda": env["CUDA_VISIBLE_DEVICES"] = ""
    rc = run_logged(cmd, d, lf, timeout_s, env)
    if not Path(outp + "_preds.npy").exists(): log(f"  ST-SHN {c} seed {seed}: no epoch finished (exit {rc}); see {lf.name}"); return None
    P = np.load(outp + "_preds.npy"); H = json.load(open(outp + "_hist.json"))            # P: (epochs, cells, days, K)
    vm = np.array([h["val"]["MAPE"] for h in H]); tm = np.array([h["test"]["MAPE"] for h in H])
    e_val, e_tst = int(np.argmin(vm)), int(np.argmin(tm))
    info = dict(model="ST-SHN", city=c, seed=seed, epochs_run=len(H), epochs_planned=epochs or 10, selected_epoch_validation=e_val + 1,
                selected_epoch_test=e_tst + 1, minutes=(time.time() - t) / 60, finished=(rc == 0))
    log(f"  ST-SHN {c} seed {seed}: {len(H)} epochs, epoch by validation {e_val + 1}, by test (official) {e_tst + 1}, {info['minutes']:.0f} min")
    return np.transpose(P[e_val], (1, 0, 2)), np.transpose(P[e_tst], (1, 0, 2)), info

# ---------------------------------------------------------------- Graph WaveNet / MTGNN (official model code, ST-HSL training rule)
def load_module(path, name, extra_path=None):
    import importlib.util
    if extra_path and str(extra_path) not in sys.path: sys.path.insert(0, str(extra_path))
    spec = importlib.util.spec_from_file_location(name, path); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
def grid_adj(R, Cc):
    N = R * Cc; A = np.eye(N, dtype=np.float32)
    for i in range(R):
        for j in range(Cc):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if 0 <= i + di < R and 0 <= j + dj < Cc: A[i * Cc + j, (i + di) * Cc + (j + dj)] = 1
    return A
def make_graph_model(kind, N, K, R, Cc):
    if kind == "Graph WaveNet":
        gd = REPOS / "Graph-WaveNet"; mp = gd / "model_patched.py"
        if not mp.exists(): mp.write_text((gd / "model.py").read_text().replace("nn.Conv1d(", "nn.Conv2d("))
        gw = load_module(mp, "gwn_model_patched")
        A = grid_adj(R, Cc); P = A / A.sum(1, keepdims=True)
        sup = [torch.tensor(P, device=DEVICE), torch.tensor(P.T.copy(), device=DEVICE)]
        return gw.gwnet(DEVICE, N, dropout=0.3, supports=sup, gcn_bool=True, addaptadj=True, aptinit=None, in_dim=K, out_dim=K,
                        residual_channels=32, dilation_channels=32, skip_channels=256, end_channels=512).to(DEVICE)
    mt = load_module(REPOS / "MTGNN" / "net.py", "mtgnn_net", REPOS / "MTGNN")
    return mt.gtnet(True, True, 2, N, DEVICE, predefined_A=None, static_feat=None, dropout=0.3, subgraph_size=min(20, N), node_dim=40,
                    dilation_exponential=1, conv_channels=32, residual_channels=32, skip_channels=64, end_channels=128, seq_length=30,
                    in_dim=K, out_dim=K, layers=3, propalpha=0.05, tanhalpha=3, layer_norm_affline=True).to(DEVICE)
def train_graph_model(kind, c, seed, max_epochs=2 if FAST else 100, patience=10, batch=16, L=30):
    set_seed(seed); Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; N = R * Cc
    S = Xall.reshape(N, T, K); mean, std = float(Xall[:, :, :ntr].mean()), float(Xall[:, :, :ntr].std())
    Z = torch.tensor((S - mean) / std, device=DEVICE); Y = torch.tensor(S, device=DEVICE)
    model = make_graph_model(kind, N, K, R, Cc); opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    back = torch.arange(L, 0, -1, device=DEVICE)
    def fwd(days):
        win = days[:, None] - back[None]                                        # (B, L): days d-L .. d-1
        x = Z[:, win, :].permute(1, 3, 0, 2)                                    # (B, K, N, L)
        out = model(x)[..., -1]                                                 # (B, K, N)
        return out.permute(0, 2, 1) * std + mean                                # (B, N, K) counts
    def predict(days):
        model.eval(); out = []
        with torch.no_grad():
            for s in range(0, len(days), 32): out.append(fwd(days[s:s + 32]).cpu().numpy())
        return np.concatenate(out)
    trd = torch.arange(L, ntr, device=DEVICE); vad = torch.arange(ntr, ntr + nva, device=DEVICE); ted = torch.arange(ntr + nva, T, device=DEVICE)
    yv = S[:, ntr:ntr + nva].transpose(1, 0, 2); best, best_state, bad, t = np.inf, None, 0, time.time()
    for ep in range(max_epochs):
        model.train(); perm = trd[torch.randperm(len(trd), device=DEVICE)]
        for s in range(0, len(perm), batch):
            b = perm[s:s + batch]; loss = F.mse_loss(fwd(b), Y[:, b, :].permute(1, 0, 2))
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        pv = predict(vad); pos = yv > 0; e = pv - yv
        crit = np.sqrt((e[pos] ** 2).mean()) + np.abs(e[pos]).mean()           # ST-HSL rule: validation RMSE + MAE on crime days
        if crit < best - 1e-5: best, bad, best_state, best_ep = crit, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep + 1
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    info = dict(model=kind, city=c, seed=seed, epochs_run=ep + 1, epochs_planned=max_epochs, selected_epoch=best_ep, selection="validation (ST-HSL rule)",
                minutes=(time.time() - t) / 60, finished=True)
    log(f"  {kind} {c} seed {seed}: {ep + 1} epochs, best {best_ep}, {info['minutes']:.1f} min")
    return predict(ted), predict(vad), info

# ---------------------------------------------------------------- metrics under the three protocols
def p_metrics(p, y, cat, K, keep):
    """MAE / MAPE per category on the cell-days in `keep` (the official codes use keep = y > 0, ST-SHN also restricts cells)."""
    out = {}
    for k in range(K):
        m = keep & (cat == k); out[f"MAE_{k}"] = np.abs(p - y)[m].mean(); out[f"MAPE_{k}"] = (np.abs(p - y) / np.maximum(y, 1e-8))[m].mean()
    out["MAE"] = np.abs(p - y)[keep].mean(); out["MAPE"] = (np.abs(p - y) / np.maximum(y, 1e-8))[keep].mean(); return out
def full_metrics(p, y, cat, K):
    out = dict(MAE_all=np.abs(p - y).mean(), RMSE_all=np.sqrt(((p - y) ** 2).mean()), deviance_all=pdev(y, np.maximum(p, 0.01)),
               MAE_zero_days=np.abs(p - y)[y == 0].mean(), MAE_crime_days=np.abs(p - y)[y > 0].mean())
    for k in range(K): out[f"deviance_{k}"] = pdev(y[cat == k], np.maximum(p[cat == k], 0.01)); out[f"MAE_all_{k}"] = np.abs(p - y)[cat == k].mean()
    return out

## Part VII. Protocol census of public crime-prediction code
Each repository is cloned at a pinned commit and its released code is searched for the lines that decide (a) which cells the test metric scores, (b) which cells the training loss uses, (c) which split chooses the reported epoch, and (d) whether the crime data and the crime pipeline are released. *Direct* evidence = the crime code or crime config itself; *indirect* = the shared metric code with the shipped (traffic) config, when the crime pipeline was not released.

In [ ]:
# Which public crime-prediction codebases score only the non-zero cells? A static audit of the released code, at pinned commits.
# Venue and year come from the papers' own README / citation; check them against the papers before quoting.
CENSUS_REPOS = {
    "ST-SHN":      dict(url="https://github.com/akaxlh/ST-SHN", commit="0a5f1adf907e91212a0b0e252a59fb3e379a975b", paper="Xia et al., IJCAI 2021", task="regression"),
    "ST-HSL":      dict(url="https://github.com/LZH-YS1998/STHSL", commit="6f3429c07f88ada44b5c34a0a098e95eedf9cbd6", paper="Li et al., ICDE 2022", task="regression"),
    "HAGEN":       dict(url="https://github.com/Rafa-zy/HAGEN", commit="23d4db95077e18b82375a8337fd8238d5ce3f958", paper="Wang et al., HAGEN (crime occurrence)", task="classification"),
    "CL4ST":       dict(url="https://github.com/HKUDS/CL4ST", commit="cebb2ddea8ef67f6e382adf306faaad1552d817a", paper="Tang et al., CIKM 2023", task="regression"),
    "STExplainer": dict(url="https://github.com/HKUDS/STExplainer", commit="23b0db0248d35b1388e42b13588b385bfa14de4d", paper="Tang et al., STExplainer", task="regression"),
    "EasyST":      dict(url="https://github.com/HKUDS/EasyST", commit="4621c2a472f795df81220e048b4cbbcf10cc1422", paper="Tang et al., CIKM 2024", task="regression"),
    "ST-HHOL":     dict(url="https://github.com/777Rebecca/ST-HHOL", commit="5d9071099013d21369a692cd6e4a036521e75304", paper="ST-HHOL, ICLR 2026", task="regression"),
}
# (aspect, file, regex, meaning when the pattern is found)
CENSUS_CHECKS = {
    "ST-SHN": [("scored cells (test metric)", "HG_ST_labcode.py", r"mask = self\.handler\.tstLocs \* \(label > 0\)", "crime cell-days only (y > 0), and only cells active on 20–80 % of days"),
               ("training loss", "HG_ST_labcode.py", r"negRate = 0", "crime cell-days only"),
               ("epoch reported", "HG_ST_labcode.py", r"reses = self\.testEpoch\(self\.handler\.tstT.*\n.*bestRes\['MAPE'\] > reses\['MAPE'\]", "best TEST MAPE"),
               ("crime data released", "Datasets/NYC_crime/trn.pkl", None, "yes (NYC, CHI)")],
    "ST-HSL": [("scored cells (test metric)", "engine.py", r"mask = 1 \* \(label > 0\)", "crime cell-days only (y > 0)"),
               ("training loss", "engine.py", r"retLabels = \(label >= 0\) \* 1", "all cell-days"),
               ("epoch reported", "train.py", r"val_metrics = res_eval\['RMSE'\] \+ res_eval\['MAE'\]", "best validation RMSE + MAE"),
               ("crime data released", "Datasets/NYC_crime/trn.pkl", None, "yes (NYC, CHI)")],
    "HAGEN": [("scored cells (test metric)", "HAGEN-code/model/hagen_supervisor.py", r"f1_score\(y_truth_reshape", "all cells (macro/micro F1 of occurrence)"),
              ("training loss", "HAGEN-code/model/loss.py", r"lossfn = nn\.BCELoss\(\)", "binary cross-entropy on all cells"),
              ("epoch reported", "HAGEN-code/model/hagen_supervisor.py", r"best_performance = valmicro \* 0\.4 \+ valmacro \* 0\.6", "best validation F1"),
              ("crime data released", "HAGEN-code/crime-data/CRIME-CHICAGO/8/test.npz", None, "yes (Chicago, LA)")],
    "CL4ST": [("scored cells (test metric)", "componenets/metrics.py", r"mask = torch\.gt\(true, mask_value\)", "cells with y > 0 when mae_thresh = 0.0 (shipped config)"),
              ("shipped config masks zeros", "config/CL4ST_pems4.yaml", r"mae_thresh: 0\.0", "mae_thresh = 0.0"),
              ("training loss", "config/CL4ST_pems4.yaml", r"criterion: 'Smooth'", "smooth L1 on all cells"),
              ("epoch reported", "train.py", r"engine\.validation\(epoch, val_loader", "validation"),
              ("crime pipeline released", "data/DataHandler_st.py", r"[Cc]rime", "crime loader in the code")],
    "STExplainer": [("scored cells (test metric)", "componenets/metrics.py", r"mask = torch\.gt\(true, mask_value\)", "cells with y > 0 when mae_thresh = 0.0 (shipped config)"),
                    ("shipped config masks zeros", "config/STExplainer_pems4.yaml", r"mae_thresh: 0\.0", "mae_thresh = 0.0"),
                    ("epoch reported", "train.py", r"engine\.validation\(epoch, val_loader", "validation"),
                    ("crime pipeline released", "data/DataHandler_st.py", r"[Cc]rime", "crime loader in the code")],
    "EasyST": [("scored cells (test metric)", "componenets/metrics.py", r"mask = torch\.gt\(true, mask_value\)", "cells with y > 0 when mae_thresh = 0.0 (shipped config)"),
               ("shipped crime config masks zeros", "config/teacher/stgcn/chi_crime/teacher_stgcn_end_dim_128_chi.yaml", r"mae_thresh: 0\.0", "mae_thresh = 0.0 in the Chicago crime config"),
               ("training loss", "config/teacher/stgcn/chi_crime/teacher_stgcn_end_dim_128_chi.yaml", r"criterion: MSE", "MSE on all cells"),
               ("crime pipeline released", "data/DataHandler_st.py", r"CHI_Crime", "crime loader in the code"),
               ("crime data released", "README.md", r"huggingface\.co/datasets/.*CHI_Crime", "external (Hugging Face), a different 42 × 35 grid")],
    "ST-HHOL": [("scored cells (test metric)", "utils/metrics.py", r"mask = \(labels != null_val\)", "cells with y ≠ 0 (null_val = 0)"),
                ("metric called with null_val = 0", "utils/engine.py", r"mask_value = torch\.tensor\(0\)", "null_val = 0"),
                ("training loss", "main.py", r"loss_fn = masked_mae", "masked MAE: crime cells only"),
                ("epoch reported", "utils/engine.py", r"mvalid_loss < min_loss", "validation"),
                ("crime data released", "data", None, "dataset folder in the repository")],
}
def clone_at(name, url, commit):
    dest = REPOS / f"census_{name}"
    if not dest.exists():
        r = subprocess.run(["git", "clone", "-q", url, str(dest)], capture_output=True, text=True)
        if r.returncode != 0: raise RuntimeError(r.stderr[-300:])
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=dest, capture_output=True, text=True).stdout.strip()
    if head != commit:
        r = subprocess.run(["git", "checkout", "-q", commit], cwd=dest, capture_output=True, text=True)
        status = "pinned commit" if r.returncode == 0 else f"pinned commit not found, HEAD {head[:10]} used"
    else: status = "pinned commit (= HEAD)"
    return dest, subprocess.run(["git", "rev-parse", "HEAD"], cwd=dest, capture_output=True, text=True).stdout.strip(), status

with part("VII (v3). protocol census of public crime-prediction code"):
    CEN = []
    for name, meta in CENSUS_REPOS.items():
        try: root, head, status = clone_at(name, meta["url"], meta["commit"])
        except Exception as e:
            CEN.append(dict(code=name, paper=meta["paper"], task=meta["task"], aspect="repository", finding="NOT AVAILABLE", verified=False, file="", commit="", commit_status=str(e)[:150])); continue
        for aspect, file, rx, meaning in CENSUS_CHECKS[name]:
            p = root / file
            if rx is None: ok = p.exists() and (p.is_file() or any(p.iterdir()))
            else: ok = p.is_file() and re.search(rx, p.read_text(errors="ignore")) is not None
            neg = "no" if aspect.startswith(("crime data released", "crime pipeline released")) else "not found in the code"
            CEN.append(dict(code=name, paper=meta["paper"], task=meta["task"], aspect=aspect, finding=meaning if ok else neg, verified=bool(ok), file=file,
                            commit=head, commit_status=status))
    CEN = save_table(pd.DataFrame(CEN), "g01_v3_protocol_census_long")
    WIDE = CEN.pivot_table(index=["code", "paper", "task"], columns="aspect", values="finding", aggfunc="first").reset_index()
    def zero_masked(code):
        g = CEN[(CEN["code"] == code) & (CEN["aspect"] == "scored cells (test metric)")]
        return bool(len(g) and g["verified"].iloc[0] and ("y > 0" in g["finding"].iloc[0] or "y ≠ 0" in g["finding"].iloc[0]))
    WIDE["test metric ignores zero cells"] = WIDE["code"].map(zero_masked)
    def evidence(code):
        g = CEN[(CEN["code"] == code) & (CEN["aspect"] == "crime pipeline released")]
        return "indirect: metric code + shipped (traffic) config; crime pipeline not released" if len(g) and not g["verified"].iloc[0] else "direct: crime code / crime config"
    WIDE["evidence"] = WIDE["code"].map(evidence)
    WIDE = save_table(WIDE, "g02_v3_protocol_census"); display(WIDE)
    reg = WIDE[WIDE["task"] == "regression"]
    log(f"  protocol census: {int(reg['test metric ignores zero cells'].sum())}/{len(reg)} regression codebases score only non-zero cells")

## Part II.1 City data (as in v1), cut at v1's last week
The panels are downloaded again and cut at v1's last week, so v3 tests on the same weeks as v1. The portals revise old records slightly; `c02b` reports the change.

In [ ]:
SOURCES = {
    "CHI": dict(api="https://data.cityofchicago.org/resource/ijzp-q8t2.csv", date="date", first_year=2001,
                cols="date,primary_type,community_area,latitude,longitude"),
    "NYC": dict(api="https://data.cityofnewyork.us/resource/qgea-i56i.csv", date="cmplnt_fr_dt", first_year=2006,
                cols="cmplnt_fr_dt,ofns_desc,addr_pct_cd,latitude,longitude"),
    "NYC_YTD": dict(api="https://data.cityofnewyork.us/resource/5uac-w243.csv", date="cmplnt_fr_dt", first_year=2006,
                    cols="cmplnt_fr_dt,ofns_desc,addr_pct_cd,latitude,longitude"),
    "SF": dict(api="https://data.sfgov.org/resource/wg3w-h783.csv", date="incident_datetime", first_year=2018,
               cols="incident_datetime,incident_id,incident_category,analysis_neighborhood,latitude,longitude"),
}
CITY_CSV = {c: os.environ.get(f"OA_{c}_CSV", "") for c in CITIES}       # optional: your own extract with the API column names

def soda_download(key, dest, start_year, end_year=None, page=500_000, select=True):
    import urllib.request, urllib.parse, urllib.error
    src = SOURCES[key]; tmp = Path(str(dest) + ".part"); total, wrote_header = 0, False
    end_year = end_year or pd.Timestamp.today().year
    with open(tmp, "wb") as out:
        for yr in range(max(start_year, src["first_year"]), end_year + 1):
            offset, size, n_year = 0, page, 0
            while True:
                q = ({"$select": src["cols"]} if select else {}) | {"$where": f"{src['date']} between '{yr}-01-01T00:00:00' and '{yr}-12-31T23:59:59'",
                     "$order": ":id", "$limit": size, "$offset": offset}
                url = src["api"] + "?" + urllib.parse.urlencode(q); data, last = None, None
                for attempt in range(5):
                    try:
                        with urllib.request.urlopen(url, timeout=900) as r: data = r.read()
                        break
                    except urllib.error.HTTPError as e:
                        last = e
                        if e.code == 400: break                       # bad query (e.g. a renamed column): no point retrying
                        time.sleep(15 * (attempt + 1))
                    except Exception as e:
                        last = e; time.sleep(15 * (attempt + 1))
                if data is None: raise RuntimeError(f"{key}: request failed for {yr}: {last!r}")
                head, _, body = data.partition(b"\n"); n = body.count(b"\n") + (0 if (body.endswith(b"\n") or not body) else 1)
                if not wrote_header: out.write(head.strip() + b"\n"); wrote_header = True
                if body: out.write(body if body.endswith(b"\n") else body + b"\n")
                n_year += n; offset += n
                if n == 0: break
                if n < size:
                    if n in (1000, 50000): size = n; continue
                    break
            log(f"  {key} {yr}: {n_year:,} rows"); total += n_year
    if total == 0: raise RuntimeError(f"{key}: the portal returned no rows")
    tmp.replace(dest); return total
def get_file(key):
    if CITY_CSV.get(key) and Path(CITY_CSV[key]).exists(): return Path(CITY_CSV[key])
    dest = DATA / f"{key.lower()}_{START_YEAR}.csv"
    if not dest.exists():
        log(f"{key}: downloading from {START_YEAR} ...")
        try: soda_download(key, dest, START_YEAR)
        except RuntimeError as e:
            if "400" not in str(e): raise
            log(f"{key}: column selection refused by the portal ({str(e)[:120]}); downloading all columns instead"); soda_download(key, dest, START_YEAR, select=False)
    return dest

FINE = ["THEFT", "BURGLARY", "VEHICLE_THEFT", "DAMAGE", "FRAUD", "ASSAULT", "ROBBERY", "SEX_OFFENSE", "DRUGS", "WEAPONS", "OTHER"]
def map_offence(s):
    s = str(s).upper().strip()
    if "MOTOR VEHICLE" in s and ("THEFT" in s or "LARCENY" in s): return "VEHICLE_THEFT"
    if "ROBBERY" in s: return "ROBBERY"
    if "BURGLAR" in s: return "BURGLARY"
    if "SEX" in s or "RAPE" in s: return "SEX_OFFENSE"
    if "FRAUD" in s or "FORGERY" in s or "DECEPTIVE" in s or "EMBEZZLEMENT" in s: return "FRAUD"
    if any(k in s for k in ("ASSAULT", "BATTERY", "HOMICIDE", "MURDER", "KIDNAP", "MANSLAUGHTER")): return "ASSAULT"
    if "LARCENY" in s or s == "THEFT" or s.startswith("THEFT OF"): return "THEFT"
    if "DAMAGE" in s or "MISCHIEF" in s or "ARSON" in s or "VANDAL" in s: return "DAMAGE"
    if "NARCOTIC" in s or "DRUG" in s or "CANNABIS" in s: return "DRUGS"
    if "WEAPON" in s or "CONCEALED CARRY" in s: return "WEAPONS"
    return "OTHER"
BOX = {"CHI": (41.60, 42.10, -87.95, -87.50), "NYC": (40.45, 40.95, -74.30, -73.65), "SF": (37.65, 37.85, -122.55, -122.33)}

def load_city(c):
    t = time.time()
    if c == "CHI":
        df = pd.read_csv(get_file("CHI"), dtype=str)
        ts = pd.to_datetime(df["date"].str.slice(0, 19), format="%Y-%m-%dT%H:%M:%S", errors="coerce")
        bad = ts.isna()
        if bad.any(): ts[bad] = pd.to_datetime(df.loc[bad, "date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
        area = pd.to_numeric(df["community_area"], errors="coerce"); area = area.where((area >= 1) & (area <= 77))
        area = area.astype("Int64").astype(str).where(area.notna()); off = df["primary_type"]
    elif c == "NYC":
        frames = [pd.read_csv(get_file("NYC"), dtype=str)]
        try:
            ytd = DATA / "nyc_ytd.csv"
            if not ytd.exists() and not CITY_CSV.get("NYC"):
                yr = pd.Timestamp.today().year; soda_download("NYC_YTD", ytd, yr - 1, yr)
            if ytd.exists():
                y = pd.read_csv(ytd, dtype=str); d0 = pd.to_datetime(frames[0]["cmplnt_fr_dt"].str.slice(0, 10), errors="coerce").max()
                y = y[pd.to_datetime(y["cmplnt_fr_dt"].str.slice(0, 10), errors="coerce") > d0]; frames.append(y); log(f"  NYC year-to-date rows added: {len(y):,}")
        except Exception as e: log(f"  NYC year-to-date not used: {str(e)[:150]}")
        df = pd.concat(frames, ignore_index=True)
        d = df["cmplnt_fr_dt"].fillna("").str.slice(0, 10)
        d = d.where(~d.str.contains("/"), pd.to_datetime(d, format="%m/%d/%Y", errors="coerce").dt.strftime("%Y-%m-%d"))
        ts = pd.to_datetime(d, format="%Y-%m-%d", errors="coerce")
        area = pd.to_numeric(df["addr_pct_cd"], errors="coerce").replace({116: 105}); area = area.where((area >= 1) & (area <= 123))
        area = area.astype("Int64").astype(str).where(area.notna()); off = df["ofns_desc"]
    else:
        df = pd.read_csv(get_file("SF"), dtype=str)
        n0 = len(df); df = df.drop_duplicates(subset=["incident_id", "incident_category"]); log(f"  SF: {n0 - len(df):,} duplicate incident × category rows removed")
        ts = pd.to_datetime(df["incident_datetime"].str.slice(0, 19), format="%Y-%m-%dT%H:%M:%S", errors="coerce")
        bad = ts.isna()
        if bad.any(): ts[bad] = pd.to_datetime(df.loc[bad, "incident_datetime"], errors="coerce")
        area = df["analysis_neighborhood"].where(df["analysis_neighborhood"].notna() & (df["analysis_neighborhood"].str.strip() != "")); off = df["incident_category"]
    lat = pd.to_numeric(df["latitude"], errors="coerce"); lon = pd.to_numeric(df["longitude"], errors="coerce")
    b = BOX[c]; okc = lat.between(b[0], b[1]) & lon.between(b[2], b[3]); lat, lon = lat.where(okc), lon.where(okc)
    keep = ts.notna() & (ts.dt.year >= START_YEAR) & (ts <= pd.Timestamp.today()) & area.notna()
    ev = pd.DataFrame({"ts": ts, "area": area, "offence_raw": off.fillna("UNKNOWN").str.upper().str.strip(), "lat": lat.astype("float32"), "lon": lon.astype("float32")})[keep.values]
    log(f"{CITY_NAME[c]}: {len(df):,} rows read, {len(ev):,} kept ({time.time() - t:.0f}s)")
    return ev.reset_index(drop=True)

def area_sort_key(a): return (0, int(a)) if str(a).isdigit() else (1, str(a))
PANEL, GEO, MAPTAB = {}, {}, []
with part("II.1 city data and weekly panels"):
    for c in CITIES:
        try: ev = load_city(c)
        except Exception as e: log(f"{c} not available, continuing without it: {str(e)[:300]}"); continue
        om = {s: map_offence(s) for s in ev["offence_raw"].unique()}
        MAPTAB.append(ev.groupby("offence_raw").size().rename("n").reset_index().assign(city=c, type=lambda d: d["offence_raw"].map(om)))
        ev["type"] = ev["offence_raw"].map(om).map({f: i for i, f in enumerate(FINE)}).astype(np.int8)
        ev["date"] = ev["ts"].dt.normalize(); ev["week"] = ev["date"] - pd.to_timedelta(ev["date"].dt.dayofweek, unit="D")
        first = ev["week"].min() + pd.Timedelta(days=7); last = ev["week"].max() - pd.Timedelta(days=7 * DROP_LAST_WEEKS[c])
        ev = ev[(ev["week"] >= first) & (ev["week"] <= last)]
        weeks = pd.date_range(first, last, freq="7D"); areas = sorted(ev["area"].astype(str).unique(), key=area_sort_key)
        wi = ((ev["week"] - weeks[0]).dt.days // 7).values; ai = pd.Categorical(ev["area"].astype(str), categories=areas).codes
        C = np.zeros((len(weeks), len(areas), len(FINE)), np.int32); np.add.at(C, (wi, ai, ev["type"].values), 1)
        cent = ev[ev["lat"].notna()].groupby("area")[["lat", "lon"]].median().reindex(areas)
        PANEL[c] = dict(weeks=weeks, areas=areas, C=C, cent=cent.fillna(cent.mean()))
        g = ev[ev["lat"].notna() & (ev["date"] > ev["date"].max() - pd.Timedelta(days=730))]
        GEO[c] = g.sample(min(len(g), 300_000), random_state=SEED)[["lat", "lon", "area"]].reset_index(drop=True)   # for mapping census tracts to areas
        log(f"{CITY_NAME[c]}: {len(weeks)} weeks ({weeks[0].date()} – {weeks[-1].date()}) × {len(areas)} areas × {len(FINE)} types, {C.sum():,} incidents")
        del ev, g; gc.collect()
    CITIES_OK = [c for c in CITIES if c in PANEL]
    if MAPTAB: save_table(pd.concat(MAPTAB).sort_values(["city", "n"], ascending=[True, False]), "c01_offence_mapping_v3")
    DS = save_table(pd.DataFrame([dict(city=c, first_week=PANEL[c]["weeks"][0].date(), last_week=PANEL[c]["weeks"][-1].date(), weeks=len(PANEL[c]["weeks"]),
                                       areas=len(PANEL[c]["areas"]), incidents=int(PANEL[c]["C"].sum()),
                                       mean_weekly_count_per_series=float(PANEL[c]["C"].mean()), share_zero_series_weeks=float((PANEL[c]["C"] == 0).mean()))
                                  for c in CITIES_OK]), "c02_panel_summary_v3")
    display(DS)
    fig, ax = plt.subplots(figsize=(13, 4))
    for c in CITIES_OK: ax.plot(PANEL[c]["weeks"], PANEL[c]["C"].sum((1, 2)), lw=1, label=CITY_NAME[c])
    ax.axvspan(TRAIN_END, VAL_END, color="grey", alpha=.15, label="calibration block"); ax.axvline(VAL_END, color="k", ls=":")
    for y_ in ROLL_YEARS: ax.axvline(pd.Timestamp(f"{y_}-01-01"), color="tab:red", ls=":", lw=.6)
    ax.set_title("Weekly recorded incidents (red dotted = rolling-origin test years)"); ax.legend(); savefig("figC1_weekly_series_v3")

In [ ]:
with part("II.1b (v3) cut the panels at v1's last week and compare with v1"):
    CHK = []
    for c in list(PANEL):
        P_ = PANEL[c]; n0 = len(P_["weeks"])
        if c in V1_LAST_WEEK:
            keep = P_["weeks"] <= V1_LAST_WEEK[c]; P_["weeks"] = P_["weeks"][keep]; P_["C"] = P_["C"][keep]
        v1 = V1_PANEL[V1_PANEL["city"] == c].iloc[0] if V1_PANEL is not None and (V1_PANEL["city"] == c).any() else None
        CHK.append(dict(city=c, weeks_downloaded=n0, weeks_used=len(P_["weeks"]), first_week=P_["weeks"][0].date(), last_week=P_["weeks"][-1].date(), areas=len(P_["areas"]),
                        incidents_v3=int(P_["C"].sum()), incidents_v1=None if v1 is None else int(v1["incidents"]), weeks_v1=None if v1 is None else int(v1["weeks"]),
                        incident_change_pct=None if v1 is None else 100 * (P_["C"].sum() - v1["incidents"]) / v1["incidents"]))
    CHK = save_table(pd.DataFrame(CHK), "c02b_v3_panel_check_against_v1"); display(CHK)
    log("  the open-data portals revise old records (late reports, reclassifications); a change of well under 1 % is expected")

## Part II.2 Point forecasts and the learned ZINB
LightGBM-Poisson (the forecaster selected in v1), seeds 0–2, H = 1–4. **GRU-ZINB**: v1's global GRU with a zero-inflated negative-binomial head trained by ZINB likelihood — the distributional deep baseline (the idea of uncertainty-aware ZINB crime models), whose 90 % interval is its own predictive distribution.

In [ ]:
RATE_MODELS = {"lgbm": "LightGBM-Poisson", "xgb": "XGBoost-Poisson", "glm": "Poisson GLM", "gru": "GRU (global)", "stgru": "ST-GRU (graph diffusion input)",
               "ma4": "Moving average (4 w)", "ma52": "Historical average (52 w)", "ses": "Exp. smoothing (0.3)", "naive": "Last week", "snaive": "Seasonal naive (52 w)"}
LEARNED = ["lgbm", "xgb", "glm", "gru", "stgru"]; TREES = ["lgbm", "xgb"]; NEURAL = ["glm", "gru", "stgru"]; NAIVE = ["ma4", "ma52", "ses", "naive", "snaive"]

def series_tables(c):
    C = PANEL[c]["C"].astype(np.float32); W = C.shape[0]
    cs = np.concatenate([np.zeros((1,) + C.shape[1:], np.float32), np.cumsum(C, 0)])
    S = np.zeros_like(C); S[0] = C[0]
    for t in range(1, W): S[t] = 0.3 * C[t] + 0.7 * S[t - 1]
    return C, cs, S

def build_rows(c, H, origins):
    C, cs, S = SER[c]; W, A, K = C.shape; o = np.asarray(origins); n = len(o)
    def rm(w): return (cs[o + 1] - cs[np.maximum(o + 1 - w, 0)]) / w
    f = {f"lag{l}": C[o - l] for l in range(8)}
    f.update(rm4=rm(4), rm13=rm(13), rm52=rm(52))
    tgt = o + H
    f["ya"] = C[tgt - 52]; f["ya5"] = np.mean([C[np.clip(tgt - 52 + d, 0, W - 1)] for d in (-2, -1, 0, 1, 2)], 0)
    last4 = cs[o + 1] - cs[np.maximum(o - 3, 0)]
    f["area_tot4"] = np.broadcast_to(np.log1p(last4.sum(-1, keepdims=True)), (n, A, K))
    f["city_type4"] = np.broadcast_to(np.log1p(last4.sum(1, keepdims=True)), (n, A, K))
    weeks = PANEL[c]["weeks"]; tw = weeks[0] + pd.to_timedelta(7 * tgt, unit="D")
    woy = np.asarray(tw.isocalendar().week, dtype=np.float32)
    f["woy_sin"] = np.broadcast_to(np.sin(2 * np.pi * woy / 52.18)[:, None, None], (n, A, K))
    f["woy_cos"] = np.broadcast_to(np.cos(2 * np.pi * woy / 52.18)[:, None, None], (n, A, K))
    df = pd.DataFrame({k: np.asarray(v, np.float32).reshape(-1) for k, v in f.items()})
    df["area"] = pd.Categorical(np.tile(np.repeat([AREA_GLOBAL[f"{c}_{a}"] for a in PANEL[c]["areas"]], K), n), categories=range(len(AREA_GLOBAL)))
    df["type"] = pd.Categorical(np.tile(np.arange(K), n * A), categories=range(K))
    df["city"] = pd.Categorical([c] * len(df), categories=CITIES_OK)
    meta = pd.DataFrame({"city": c, "origin": np.repeat(o, A * K).astype(np.int32), "target_week": np.repeat(np.asarray(tw), A * K),
                         "area": np.tile(np.repeat(PANEL[c]["areas"], K), n), "type": np.tile(np.arange(K), n * A).astype(np.int8)})
    meta["ai"] = np.tile(np.repeat(np.arange(A), K), n).astype(np.int32)
    meta["y"] = C[tgt].reshape(-1)
    meta["naive"] = C[o].reshape(-1); meta["ma4"] = f["rm4"].reshape(-1); meta["ses"] = S[o].reshape(-1); meta["ma52"] = f["rm52"].reshape(-1)
    meta["snaive"] = np.asarray(f["ya"]).reshape(-1)
    return df, meta

# ---------------------------------------------------------------- Poisson GLM, GRU and ST-GRU (torch)
GLM_NUM = [f"lag{l}" for l in range(8)] + ["rm4", "rm13", "rm52", "ya", "ya5", "area_tot4", "city_type4", "woy_sin", "woy_cos"]
def glm_design(X):
    num = X[GLM_NUM].values.astype(np.float32); num[:, :13] = np.log1p(num[:, :13]); return num
class PoissonGLM(nn.Module):
    def __init__(self, n_num):
        super().__init__(); self.lin = nn.Linear(n_num, 1); self.ea = nn.Embedding(len(AREA_GLOBAL), 1); self.et = nn.Embedding(len(FINE), 1); self.ec = nn.Embedding(len(CITIES_OK), 1)
        for e in (self.ea, self.et, self.ec): nn.init.zeros_(e.weight)
    def forward(self, xn, a, t, c): return (self.lin(xn) + self.ea(a) + self.et(t) + self.ec(c)).squeeze(-1)
SEQ_L = 52
class SeqForecaster(nn.Module):
    def __init__(self, spatial, hid=64):
        super().__init__(); self.spatial = spatial
        self.gru = nn.GRU(4 if spatial else 3, hid, batch_first=True)
        self.ea = nn.Embedding(len(AREA_GLOBAL), 16); self.et = nn.Embedding(len(FINE), 8); self.ec = nn.Embedding(len(CITIES_OK), 4)
        self.head = nn.Sequential(nn.Linear(hid + 28 + 2, 64), nn.GELU(), nn.Linear(64, 1))
    def forward(self, seq, ag, t, c, woy):
        _, h = self.gru(seq); return self.head(torch.cat([h[-1], self.ea(ag), self.et(t), self.ec(c), woy], 1)).squeeze(-1)
def seq_batch(c, o, a, k, spatial):
    S_ = SEQ[c]; win = o[:, None] - torch.arange(SEQ_L - 1, -1, -1, device=DEVICE)[None]
    parts = [S_["x"][win, a[:, None], k[:, None]], S_["at"][win, a[:, None]], S_["ct"][win, k[:, None]]]
    if spatial: parts.append(S_["nb"][win, a[:, None], k[:, None]])
    return torch.stack(parts, -1)

def train_torch_rate(kind, X, M, fit, es, seed, epochs=3 if FAST else 20, batch=2048, lr=2e-3):
    set_seed(seed); ccode = {c: i for i, c in enumerate(CITIES_OK)}
    o = torch.as_tensor(M["origin"].values.astype(np.int64), device=DEVICE); ai = torch.as_tensor(M["ai"].values.astype(np.int64), device=DEVICE)
    ag = torch.as_tensor(X["area"].cat.codes.values.astype(np.int64), device=DEVICE); ty = torch.as_tensor(M["type"].values.astype(np.int64), device=DEVICE)
    cc = torch.as_tensor(M["city"].map(ccode).values.astype(np.int64), device=DEVICE)
    y = torch.as_tensor(M["y"].values.astype(np.float32), device=DEVICE)
    woy = torch.as_tensor(X[["woy_sin", "woy_cos"]].values.astype(np.float32), device=DEVICE)
    fit_idx = torch.as_tensor(np.where(fit)[0], device=DEVICE); es_idx = torch.as_tensor(np.where(es)[0], device=DEVICE)
    if kind == "glm":
        xn = torch.as_tensor(glm_design(X), device=DEVICE); mu_, sd_ = xn[fit_idx].mean(0), xn[fit_idx].std(0) + 1e-6
        xn = (xn - mu_) / sd_; model = PoissonGLM(xn.shape[1]).to(DEVICE)
        fwd = lambda idx: model(xn[idx], ag[idx], ty[idx], cc[idx])
    else:
        model = SeqForecaster(kind == "stgru").to(DEVICE)
        def fwd(idx):
            out = torch.empty(len(idx), device=DEVICE)
            for cn, ci in ccode.items():
                sel = (cc[idx] == ci)
                if sel.any():
                    j = idx[sel]; out[sel] = model(seq_batch(cn, o[j], ai[j], ty[j], kind == "stgru"), ag[j], ty[j], cc[j], woy[j])
            return out
    opt = torch.optim.AdamW(model.parameters(), lr=lr * (5 if kind == "glm" else 1), weight_decay=1e-4)
    def eval_dev(idx):
        model.eval(); tot = 0.0
        with torch.no_grad():
            for s in range(0, len(idx), 32768):
                b = idx[s:s + 32768]; tot += F.poisson_nll_loss(fwd(b), y[b], log_input=True, full=True, reduction="sum").item()
        return tot / len(idx)
    best, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train(); perm = fit_idx[torch.randperm(len(fit_idx), device=DEVICE)]
        for s in range(0, len(perm), batch):
            b = perm[s:s + batch]; loss = F.poisson_nll_loss(fwd(b), y[b], log_input=True, full=False)
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        v = eval_dev(es_idx)
        if v < best - 1e-5: best, bad, best_state = v, 0, {k_: t_.detach().clone() for k_, t_ in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 3: break
    model.load_state_dict(best_state); model.eval(); out = []; allidx = torch.arange(len(M), device=DEVICE)
    with torch.no_grad():
        for s in range(0, len(M), 32768): out.append(torch.exp(fwd(allidx[s:s + 32768]).clamp(max=12)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def knn_adj(c, k=5):
    xy = PANEL[c]["cent"]; X_ = np.c_[xy["lat"].values * 110.57, xy["lon"].values * 111.32 * np.cos(np.radians(xy["lat"].mean()))]
    D = ((X_[:, None] - X_[None]) ** 2).sum(-1); np.fill_diagonal(D, np.inf); A = len(X_); nb = np.zeros((A, A), np.float32)
    for i, js in enumerate(np.argsort(D, 1)[:, :k]): nb[i, js] = 1
    return nb / nb.sum(1, keepdims=True)

In [ ]:
# ------------------------------------------------------------------ zero-inflated negative binomial (ZINB) helpers
# y ~ ZINB(pi, m, r): P(0) = pi + (1 - pi) * NB(0; m, r), P(k > 0) = (1 - pi) * NB(k; m, r); NB with mean m and size r; E[y] = (1 - pi) * m
from scipy.optimize import minimize
from scipy.special import gammaln, expit
def zinb_logpmf(y, pi, m, r):
    y = np.asarray(y, float); m = np.maximum(np.asarray(m, float), 1e-8); r = np.asarray(r, float); pi = np.clip(np.asarray(pi, float), 1e-8, 1 - 1e-8)
    lr = np.log(r / (r + m)); lm = np.log(m / (r + m))
    nb = gammaln(y + r) - gammaln(r) - gammaln(y + 1) + r * lr + y * lm
    return np.where(y == 0, np.logaddexp(np.log(pi), np.log1p(-pi) + r * lr), np.log1p(-pi) + nb)
def zinb_fit(mu, y):
    """(pi, r) by maximum likelihood on held-out data, with the zero-inflated mean tied to the forecast: (1 - pi) * m = mu."""
    mu = np.maximum(np.asarray(mu, float), 1e-6); y = np.asarray(y, float)
    def nll(th):
        pi = expit(th[0]); r = np.exp(np.clip(th[1], -6, 9)); return -zinb_logpmf(y, pi, mu / (1 - pi), r).mean()
    best = None
    for p0 in (-4.0, -1.5, 0.0):                                    # a few starts: the likelihood can be flat in pi
        res = minimize(nll, x0=np.array([p0, 1.0]), method="Nelder-Mead", options=dict(xatol=1e-4, fatol=1e-7, maxiter=800))
        if best is None or res.fun < best.fun: best = res
    return float(expit(best.x[0])), float(np.exp(np.clip(best.x[1], -6, 9)))
def zinb_ppf(q, pi, m, r):
    """Smallest k with F(k) >= q."""
    q = np.asarray(q, float); pi = np.asarray(pi, float); m = np.maximum(np.asarray(m, float), 1e-8); r = np.asarray(r, float)
    qq = (q - pi) / (1 - pi)
    k = _nb.ppf(np.clip(qq, 1e-12, 1 - 1e-12), r, r / (r + m))
    return np.where(qq <= 0, 0.0, np.maximum(k, 0.0))
def zinb_interval(mu, pi, r):
    """Equal-tailed 90 % interval of the ZINB whose mean is the forecast mu."""
    mu = np.maximum(np.asarray(mu, float), 1e-6); pi = np.broadcast_to(np.asarray(pi, float), mu.shape); m = mu / (1 - pi)
    return zinb_ppf(ALPHA / 2, pi, m, r), zinb_ppf(1 - ALPHA / 2, pi, m, r)
def zinb_methods(mu_cal, y_cal, mu_t, k=5):
    """Two parametric ZINB baselines on top of the point forecast: one (pi, r) for all series, and one per volume bin (the Mondrian bins)."""
    out, par = {}, []
    pi, r = zinb_fit(mu_cal, y_cal); out["ZINB (fitted)"] = zinb_interval(mu_t, pi, r); par.append(dict(variant="global", bin=-1, pi=pi, r=r, n=len(mu_cal)))
    e = mondrian_edges(mu_cal, k); bc, bt = np.digitize(mu_cal, e), np.digitize(mu_t, e); lo = np.zeros(len(mu_t)); hi = np.zeros(len(mu_t))
    for b in range(len(e) + 1):
        mc_, mt_ = bc == b, bt == b
        if not mt_.any(): continue
        pb, rb = zinb_fit(mu_cal[mc_], y_cal[mc_]) if mc_.sum() >= 30 else (pi, r)
        lo[mt_], hi[mt_] = zinb_interval(mu_t[mt_], pb, rb); par.append(dict(variant="per volume bin", bin=b, pi=pb, r=rb, n=int(mc_.sum())))
    out["ZINB (fitted per volume bin)"] = (lo, hi)
    return out, par

In [ ]:
# ---------------------------------------------------------------- learned ZINB forecaster (GRU with a ZINB head), the distributional deep baseline
class SeqZINB(SeqForecaster):
    """Same encoder as the GRU forecaster of v1; the head outputs (log m, logit pi, log r) of a ZINB instead of one Poisson rate."""
    def __init__(self, hid=64):
        super().__init__(False, hid); self.head = nn.Sequential(nn.Linear(hid + 28 + 2, 64), nn.GELU(), nn.Linear(64, 3))
    def forward(self, seq, ag, t, c, woy):
        _, h = self.gru(seq); return self.head(torch.cat([h[-1], self.ea(ag), self.et(t), self.ec(c), woy], 1))
def zinb_nll_torch(out, y):
    lm, lp, lr = out[:, 0].clamp(-8, 8), out[:, 1], out[:, 2].clamp(-5, 6); m, r = lm.exp(), lr.exp()
    log_pi, log_1mpi = F.logsigmoid(lp), F.logsigmoid(-lp); t_r = r * (lr - torch.logaddexp(lr, lm))       # r * log(r / (r + m))
    nb = torch.lgamma(y + r) - torch.lgamma(r) - torch.lgamma(y + 1) + t_r + y * (lm - torch.logaddexp(lr, lm))
    ll = torch.where(y == 0, torch.logaddexp(log_pi, log_1mpi + t_r), log_1mpi + nb)
    return -ll
def train_gru_zinb(X, M, fit, es, seed, epochs=3 if FAST else 20, batch=2048, lr=2e-3):
    """Returns (mu, pi, m, r) for every row of M; trained on `fit`, early-stopped on the ZINB NLL of `es` (same recipe as v1's GRU)."""
    set_seed(seed); ccode = {c: i for i, c in enumerate(CITIES_OK)}
    o = torch.as_tensor(M["origin"].values.astype(np.int64), device=DEVICE); ai = torch.as_tensor(M["ai"].values.astype(np.int64), device=DEVICE)
    ag = torch.as_tensor(X["area"].cat.codes.values.astype(np.int64), device=DEVICE); ty = torch.as_tensor(M["type"].values.astype(np.int64), device=DEVICE)
    cc = torch.as_tensor(M["city"].map(ccode).values.astype(np.int64), device=DEVICE); y = torch.as_tensor(M["y"].values.astype(np.float32), device=DEVICE)
    woy = torch.as_tensor(X[["woy_sin", "woy_cos"]].values.astype(np.float32), device=DEVICE)
    fit_idx = torch.as_tensor(np.where(fit)[0], device=DEVICE); es_idx = torch.as_tensor(np.where(es)[0], device=DEVICE)
    model = SeqZINB().to(DEVICE)
    with torch.no_grad(): model.head[-1].bias.copy_(torch.tensor([0.0, -2.0, 1.0]))   # start near a Poisson-like NB with few structural zeros
    def fwd(idx):
        out = torch.empty(len(idx), 3, device=DEVICE)
        for cn, ci in ccode.items():
            sel = (cc[idx] == ci)
            if sel.any(): j = idx[sel]; out[sel] = model(seq_batch(cn, o[j], ai[j], ty[j], False), ag[j], ty[j], cc[j], woy[j])
        return out
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    def eval_nll(idx):
        model.eval(); tot = 0.0
        with torch.no_grad():
            for s in range(0, len(idx), 32768): b = idx[s:s + 32768]; tot += zinb_nll_torch(fwd(b), y[b]).sum().item()
        return tot / len(idx)
    best, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train(); perm = fit_idx[torch.randperm(len(fit_idx), device=DEVICE)]
        for s in range(0, len(perm), batch):
            b = perm[s:s + batch]; loss = zinb_nll_torch(fwd(b), y[b]).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        v = eval_nll(es_idx)
        if v < best - 1e-5: best, bad, best_state = v, 0, {k_: t_.detach().clone() for k_, t_ in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 3: break
    model.load_state_dict(best_state); model.eval(); out = []; allidx = torch.arange(len(M), device=DEVICE)
    with torch.no_grad():
        for s in range(0, len(M), 32768): out.append(fwd(allidx[s:s + 32768]).cpu().numpy())
    out = np.concatenate(out).astype(np.float64)
    m = np.exp(np.clip(out[:, 0], -8, 8)); pi = expit(out[:, 1]); r = np.exp(np.clip(out[:, 2], -5, 6))
    return dict(mu=(1 - pi) * m, pi=pi, m=m, r=r)

FC = {}
with part("II.2 (v3) point forecasts for the intervals: LightGBM-Poisson (as selected in v1) and GRU-ZINB"):
    SER = {c: series_tables(c) for c in CITIES_OK}
    AREA_GLOBAL = {f"{c}_{a}": i for i, (c, a) in enumerate((c, a) for c in CITIES_OK for a in PANEL[c]["areas"])}
    SEQ = {}
    for c in CITIES_OK:
        C = PANEL[c]["C"].astype(np.float32); NBC = np.einsum("ab,wbk->wak", knn_adj(c), C)
        SEQ[c] = {k: torch.as_tensor(v, device=DEVICE) for k, v in dict(x=np.log1p(C), at=np.log1p(C.sum(-1)), ct=np.log1p(C.sum(1)), nb=np.log1p(NBC)).items()}
    ZPRED = {}                                                       # ZPRED[seed] = dict(mu, pi, m, r) for the H = 1 rows
    ACC3 = []
    for H in HORIZONS:
        X, M = [], []
        for c in CITIES_OK:
            W = PANEL[c]["C"].shape[0]; Xc, Mc = build_rows(c, H, np.arange(52, W - H)); X.append(Xc); M.append(Mc)
        X = pd.concat(X, ignore_index=True); M = pd.concat(M, ignore_index=True)
        for col in ("city", "area"): M[col] = M[col].astype("category")
        tw = M["target_week"]
        tr = (tw <= TRAIN_END).values; ca = ((tw > TRAIN_END) & (tw <= VAL_END)).values; te = (tw > VAL_END).values
        es = tr & (tw > TRAIN_END - pd.Timedelta(weeks=26)).values; fit = tr & ~es
        log(f"H={H}: {len(M):,} rows ({fit.sum():,} fit, {es.sum():,} early stop, {ca.sum():,} calibration, {te.sum():,} test)")
        for s in SEEDS_TREE:
            t = time.time(); m = fit_lgb_pois(X[fit], M["y"][fit], X[es], M["y"][es], seed=s); M[f"lgbm_s{s}"] = m.predict(X).astype(np.float32)
            log(f"  LightGBM seed {s}: {m.best_iteration_} trees, {time.time() - t:.0f}s")
        if H == 1:
            for s in SEEDS_NN:
                t = time.time(); ZPRED[s] = train_gru_zinb(X, M, fit, es, seed=s); M[f"gruzinb_s{s}"] = ZPRED[s]["mu"].astype(np.float32)
                log(f"  GRU-ZINB seed {s}: {time.time() - t:.0f}s, mean pi on test {ZPRED[s]['pi'][te].mean():.3f}, median r {np.median(ZPRED[s]['r'][te]):.2f}")
        for split, mk in (("calibration", ca), ("test", te)):
            for c in CITIES_OK:
                mm = mk & (M["city"] == c).values
                for col in [c_ for c_ in M.columns if re.fullmatch(r"(lgbm|gruzinb)_s\d+", c_)]:
                    ACC3.append(dict(H=H, split=split, city=c, model=col.split("_s")[0], seed=int(col.split("_s")[1]), poisson_deviance=pdev(M["y"].values[mm], M[col].values[mm])))
        FC[H] = dict(M=M, X=X if H in (1, 4) else None, masks=dict(fit=fit, es=es, cal=ca, test=te))
        if H not in (1, 4): del X
        gc.collect()
    SEL_H = {H: "lgbm" for H in HORIZONS}; SEL_SEEDS = {H: list(SEEDS_TREE) for H in HORIZONS}
    ACC3 = save_table(pd.DataFrame(ACC3), "c08_v3_forecast_deviance")
    display(ACC3.groupby(["H", "split", "city", "model"])["poisson_deviance"].mean().unstack("model").round(4))

## Part III. 90 % intervals with ZINB baselines
All v1 methods plus three ZINB baselines: **ZINB (fitted)** – one (π, r) fitted by maximum likelihood on the calibration block, mean tied to the forecast; **ZINB (fitted per volume bin)** – one (π, r) per Mondrian volume bin; **GRU-ZINB (learned)** – the learned distribution (H = 1).

In [ ]:
ACI_GAMMA = 0.05
def aci_intervals(mu, y, weeks, cal_scores, H, bins=None, cal_bins=None, gamma=ACI_GAMMA):
    """Adaptive conformal inference with a delay of H weeks. Optional Mondrian bins (one alpha per bin)."""
    mu = np.asarray(mu, float); y = np.asarray(y, float); s = np.sqrt(mu + 0.5)
    bins = np.zeros(len(mu), int) if bins is None else bins; cal_bins = np.zeros(len(cal_scores), int) if cal_bins is None else cal_bins
    ub = np.unique(np.r_[bins, cal_bins]); alpha = {b: ALPHA for b in ub}; sorted_cal = {b: np.sort(cal_scores[cal_bins == b]) for b in ub}
    lo = np.zeros(len(mu)); hi = np.zeros(len(mu)); uw = np.sort(np.unique(weeks)); errs = {b: {} for b in ub}
    for i, w in enumerate(uw):
        m = weeks == w
        for b in ub:
            mb = m & (bins == b)
            if not mb.any(): continue
            known = [errs[b][uw[j]] for j in range(max(0, i - 400), i - H + 1) if uw[j] in errs[b]]   # errors of weeks already observed
            a = ALPHA
            for e in known: a = a + gamma * (ALPHA - e)
            alpha[b] = a; sc = sorted_cal[b] if len(sorted_cal[b]) else np.sort(cal_scores)
            if a <= 0: q = np.inf
            elif a >= 1: q = 0.0
            else:
                n = len(sc); lev = min(1.0, np.ceil((n + 1) * (1 - a)) / n); q = float(np.quantile(sc, lev, method="higher"))
            l_, h_ = int_from_q(mu[mb], q if np.isfinite(q) else 1e6)
            lo[mb], hi[mb] = l_, h_
            errs[b][w] = float(1 - ((y[mb] >= l_) & (y[mb] <= h_)).mean())
    return lo, hi

def interval_suite(H, c, mu_cal, y_cal, mu_t, y_t, weeks_t, extra=None):
    """All interval methods for one horizon and city. `extra` holds CQR quantiles and the locally weighted spread if available."""
    out = {}
    out["Poisson"] = parametric_interval(mu_t, "Poisson"); out["Neg. binomial"] = parametric_interval(mu_t, "NB", nb_size(mu_cal, y_cal))
    sc = norm_score(y_cal, mu_cal); out["Split conformal (CP)"] = int_from_q(mu_t, conformal_q(sc))
    for k in (3, 5, 10):
        e = mondrian_edges(mu_cal, k); bc = np.digitize(mu_cal, e); qb = np.array([conformal_q(sc[bc == i]) for i in range(len(e) + 1)])
        out[f"Mondrian CP (k={k})" if k != 5 else "Mondrian CP"] = int_from_q(mu_t, qb[np.digitize(mu_t, e)])
    e5 = mondrian_edges(mu_cal, 5)
    out["ACI"] = aci_intervals(mu_t, y_t, weeks_t, sc, H)
    out["Mondrian-ACI"] = aci_intervals(mu_t, y_t, weeks_t, sc, H, bins=np.digitize(mu_t, e5), cal_bins=np.digitize(mu_cal, e5))
    if extra is not None:
        if "q_lo_cal" in extra:
            E = np.maximum(extra["q_lo_cal"] - y_cal, y_cal - extra["q_hi_cal"]); Q = conformal_q(E)
            lo = np.maximum(np.ceil(extra["q_lo_t"] - Q), 0); hi = np.floor(extra["q_hi_t"] + Q); bad = hi < lo
            out["CQR"] = (np.where(bad, np.round(mu_t), lo), np.where(bad, np.round(mu_t), hi))
        if "sig_cal" in extra:
            s_cal = np.maximum(extra["sig_cal"], 0.1); s_t = np.maximum(extra["sig_t"], 0.1)
            q = conformal_q(np.abs(y_cal - mu_cal) / s_cal); out["Locally weighted CP"] = int_from_q(mu_t, q, scale=s_t)
    return out

def series_volume_quintiles(c, train_end):
    C = PANEL[c]["C"]; v = C[PANEL[c]["weeks"] <= train_end].mean(0)
    V = pd.DataFrame({"area": np.repeat(PANEL[c]["areas"], len(FINE)), "type": np.tile(np.arange(len(FINE)), len(PANEL[c]["areas"])).astype(np.int8), "vol": v.reshape(-1)})
    V["vq"] = pd.qcut(V["vol"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int); return V

In [ ]:
def gru_zinb_interval(Z, idx):
    """Central 90 % interval of the learned ZINB (no calibration step: the model's own distribution, as distributional deep models report it)."""
    return zinb_ppf(ALPHA / 2, Z["pi"][idx], Z["m"][idx], Z["r"][idx]), zinb_ppf(1 - ALPHA / 2, Z["pi"][idx], Z["m"][idx], Z["r"][idx])
def all_intervals(H, c, mu_cal, y_cal, mu_t, y_t, weeks_t, ex, Z=None, idx_t=None):
    res = interval_suite(H, c, mu_cal, y_cal, mu_t, y_t, weeks_t, ex)
    zres, zpar = zinb_methods(mu_cal, y_cal, mu_t); res.update(zres)
    if Z is not None: res["GRU-ZINB (learned)"] = gru_zinb_interval(Z, idx_t)
    return res, zpar

COV3, ZPAR = [], []
with part("III (v3). intervals on the main split, with ZINB baselines"):
    for H in HORIZONS:
        d = FC[H]; M = d["M"]; X = d["X"]; s0 = SEL_SEEDS[H][0]; scol = f"lgbm_s{s0}"
        extra_all = None
        if X is not None:                                             # CQR and locally weighted CP (H = 1 and 4), as in v1
            fit, es, cal, te = (d["masks"][k] for k in ("fit", "es", "cal", "test")); sel = cal | te; t = time.time()
            ql = fit_lgb_quantile(X[fit], M["y"][fit], X[es], M["y"][es], ALPHA / 2); qh = fit_lgb_quantile(X[fit], M["y"][fit], X[es], M["y"][es], 1 - ALPHA / 2)
            Xr = X.assign(mu=M[scol].values); rs = fit_lgb_l2(Xr[es], np.abs(M["y"][es] - M[scol][es]), Xr[es], np.abs(M["y"][es] - M[scol][es]))
            extra_all = dict(q_lo=np.full(len(M), np.nan), q_hi=np.full(len(M), np.nan), sig=np.full(len(M), np.nan))
            extra_all["q_lo"][sel] = ql.predict(X[sel]); extra_all["q_hi"][sel] = qh.predict(X[sel]); extra_all["sig"][sel] = rs.predict(Xr[sel])
            log(f"  H={H}: CQR and locally weighted CP models in {time.time() - t:.0f}s")
        for c in CITIES_OK:
            mc = d["masks"]["cal"] & (M["city"] == c).values; mt = d["masks"]["test"] & (M["city"] == c).values
            if mc.sum() == 0 or mt.sum() == 0: continue
            T = M.loc[mt, ["area", "type", "target_week", "y", scol]].reset_index(drop=True).merge(series_volume_quintiles(c, TRAIN_END), on=["area", "type"], how="left")
            ex = None if extra_all is None else dict(q_lo_cal=extra_all["q_lo"][mc], q_hi_cal=extra_all["q_hi"][mc], q_lo_t=extra_all["q_lo"][mt], q_hi_t=extra_all["q_hi"][mt],
                                                     sig_cal=extra_all["sig"][mc], sig_t=extra_all["sig"][mt])
            res, zpar = all_intervals(H, c, M[scol].values[mc], M["y"].values[mc], T[scol].values, T["y"].values, T["target_week"].values, ex,
                                      Z=ZPRED.get(s0) if H == 1 else None, idx_t=np.where(mt)[0])
            ZPAR += [p | dict(H=H, city=c, seed=s0) for p in zpar]
            for meth, (lo, hi) in res.items():
                COV3.append(dict(H=H, city=c, method=meth, forecaster="GRU-ZINB" if meth.startswith("GRU-ZINB") else "LightGBM-Poisson", seed=s0,
                                 **coverage_report(T["y"].values, lo, hi, T["vq"].values, T["area"].values)))
        for s in SEL_SEEDS[H][1:]:                                    # other seeds: split CP, Mondrian and the ZINB baselines
            col = f"lgbm_s{s}"
            for c in CITIES_OK:
                mc = d["masks"]["cal"] & (M["city"] == c).values; mt = d["masks"]["test"] & (M["city"] == c).values
                T = M.loc[mt, ["area", "type", "y", col]].reset_index(drop=True).merge(series_volume_quintiles(c, TRAIN_END), on=["area", "type"], how="left")
                mu_c, y_c, mu_t = M[col].values[mc], M["y"].values[mc], T[col].values
                sc = norm_score(y_c, mu_c); e = mondrian_edges(mu_c); bc = np.digitize(mu_c, e); qb = np.array([conformal_q(sc[bc == i]) for i in range(len(e) + 1)])
                meths = {"Split conformal (CP)": int_from_q(mu_t, conformal_q(sc)), "Mondrian CP": int_from_q(mu_t, qb[np.digitize(mu_t, e)])}
                zres, zpar = zinb_methods(mu_c, y_c, mu_t); meths.update(zres); ZPAR += [p | dict(H=H, city=c, seed=s) for p in zpar]
                if H == 1 and s in ZPRED: meths["GRU-ZINB (learned)"] = gru_zinb_interval(ZPRED[s], np.where(mt)[0])
                for meth, (lo, hi) in meths.items():
                    COV3.append(dict(H=H, city=c, method=meth, forecaster="GRU-ZINB" if meth.startswith("GRU-ZINB") else "LightGBM-Poisson", seed=s,
                                     **coverage_report(T["y"].values, lo, hi, T["vq"].values, T["area"].values)))
        log(f"  intervals H={H} done")
    COV3 = pd.DataFrame(COV3); qc = [f"cov_Q{q}" for q in range(1, 6)]
    COV3["worst_under_coverage"] = (0.9 - COV3[qc]).max(axis=1); COV3["worst_over_coverage"] = (COV3[qc] - 0.9).max(axis=1)
    COV3 = save_table(COV3, "d03_v3_interval_coverage_with_zinb"); ZPAR = save_table(pd.DataFrame(ZPAR), "d05_v3_zinb_parameters")
    save_table(COV3.groupby(["H", "city", "method"])[["coverage", "max_abs_dev_from_90", "worst_under_coverage", "width", "interval_score"]].agg(["mean", "std", "count"]).reset_index(),
               "d04_v3_interval_coverage_over_seeds")
    show = COV3[COV3["seed"] == SEEDS_TREE[0]]
    display(show.pivot_table(index="method", columns=["city", "H"], values="worst_under_coverage").mul(100).round(1))
    display(show[show["H"] == 1][["city", "method", "coverage", "width", "interval_score", "cov_Q1", "cov_Q3", "cov_Q5", "worst_under_coverage", "worst_over_coverage"]].round(3))
    KEY_METHODS = ["Poisson", "Neg. binomial", "ZINB (fitted)", "ZINB (fitted per volume bin)", "GRU-ZINB (learned)", "Split conformal (CP)", "Mondrian CP", "Mondrian-ACI"]
    fig, ax = plt.subplots(1, len(CITIES_OK), figsize=(5.3 * len(CITIES_OK), 4.3), sharey=True); ax = np.atleast_1d(ax)
    for i, c in enumerate(CITIES_OK):
        g = show[(show["H"] == 1) & (show["city"] == c)]
        for meth in KEY_METHODS:
            r = g[g["method"] == meth]
            if len(r): ax[i].plot(range(1, 6), [100 * r.iloc[0][f"cov_Q{q}"] for q in range(1, 6)], marker="o", label=meth, lw=2.2 if meth.startswith("Mondrian") else 1,
                                  ls="-" if meth.startswith("Mondrian") else ("--" if "ZINB" in meth else ":"))
        ax[i].axhline(90, color="k", lw=.8, ls=":"); ax[i].set_xticks(range(1, 6)); ax[i].set_xticklabels(["Q1\nquiet", "Q2", "Q3", "Q4", "Q5\nbusy"])
        ax[i].set_title(f"{CITY_NAME[c]}: coverage by series volume (H = 1)")
    ax[0].set_ylabel("coverage of 90 % interval (%)"); ax[-1].legend(fontsize=7); savefig("figD3_v3_coverage_by_quintile_with_zinb")

## Part IV. Rolling origins with ZINB baselines (test years 2020–2024; GRU-ZINB re-trained per fold)

In [ ]:
ROLL3 = []
with part("IV (v3). rolling origins, with ZINB baselines"):
    M1 = FC[1]["M"]; X1 = FC[1]["X"]; tw = M1["target_week"]
    for Y in ROLL_YEARS:
        if time_left() < 1.5: log(f"  time budget: fold {Y} skipped"); continue
        t0 = pd.Timestamp(f"{Y}-01-01"); cal_start = t0 - pd.Timedelta(weeks=26)
        te = ((tw >= t0) & (tw < pd.Timestamp(f"{Y + 1}-01-01"))).values; ca = ((tw >= cal_start) & (tw < t0)).values; tr = (tw < cal_start).values
        es = tr & (tw >= cal_start - pd.Timedelta(weeks=13)).values; fit = tr & ~es
        if te.sum() == 0 or fit.sum() == 0: continue
        t = time.time(); m = fit_lgb_pois(X1[fit], M1["y"][fit], X1[es], M1["y"][es], seed=0); sel = ca | te | es
        mu = np.full(len(M1), np.nan, np.float32); mu[sel] = m.predict(X1[sel])
        ql = fit_lgb_quantile(X1[fit], M1["y"][fit], X1[es], M1["y"][es], ALPHA / 2); qh = fit_lgb_quantile(X1[fit], M1["y"][fit], X1[es], M1["y"][es], 1 - ALPHA / 2)
        Xr = X1.assign(mu=mu); rs = fit_lgb_l2(Xr[es], np.abs(M1["y"][es] - mu[es]), Xr[es], np.abs(M1["y"][es] - mu[es]))
        ex_all = dict(q_lo=np.full(len(M1), np.nan), q_hi=np.full(len(M1), np.nan), sig=np.full(len(M1), np.nan))
        s2 = ca | te; ex_all["q_lo"][s2] = ql.predict(X1[s2]); ex_all["q_hi"][s2] = qh.predict(X1[s2]); ex_all["sig"][s2] = rs.predict(Xr[s2])
        Zf = train_gru_zinb(X1, M1, fit, es, seed=0)                  # the learned ZINB, re-trained for this fold
        log(f"  fold {Y}: models in {time.time() - t:.0f}s")
        for c in CITIES_OK:
            cm = (M1["city"] == c).values
            if (tr & cm).sum() == 0 or len(np.unique(tw[tr & cm])) < 26: log(f"    {c}: fewer than 26 training weeks before {Y}, skipped"); continue
            mc, mt = ca & cm, te & cm
            if mc.sum() == 0 or mt.sum() == 0: continue
            y_t = M1["y"].values[mt]; wk = tw.values[mt]
            T = M1.loc[mt, ["area", "type"]].reset_index(drop=True).merge(series_volume_quintiles(c, cal_start), on=["area", "type"], how="left")
            ex = dict(q_lo_cal=ex_all["q_lo"][mc], q_hi_cal=ex_all["q_hi"][mc], q_lo_t=ex_all["q_lo"][mt], q_hi_t=ex_all["q_hi"][mt], sig_cal=ex_all["sig"][mc], sig_t=ex_all["sig"][mt])
            res, _ = all_intervals(1, c, mu[mc], M1["y"].values[mc], mu[mt], y_t, wk, ex, Z=Zf, idx_t=np.where(mt)[0])
            for meth, (lo, hi) in res.items():
                ROLL3.append(dict(test_year=Y, city=c, method=meth, **coverage_report(y_t, lo, hi, T["vq"].values, T["area"].values)))
        save_table(pd.DataFrame(ROLL3), "e03_v3_rolling_coverage_with_zinb")
    ROLL3 = pd.DataFrame(ROLL3)
    if len(ROLL3):
        qc = [f"cov_Q{q}" for q in range(1, 6)]
        ROLL3["worst_under_coverage"] = (0.9 - ROLL3[qc]).max(axis=1); ROLL3["worst_over_coverage"] = (ROLL3[qc] - 0.9).max(axis=1)
        ROLL3 = save_table(ROLL3, "e03_v3_rolling_coverage_with_zinb")
        display(ROLL3.pivot_table(index="method", columns=["city", "test_year"], values="max_abs_dev_from_90").mul(100).round(1))
        display(ROLL3.pivot_table(index="method", columns=["city", "test_year"], values="worst_under_coverage").mul(100).round(1))
        fig, ax = plt.subplots(1, len(CITIES_OK), figsize=(5.3 * len(CITIES_OK), 4), sharey=True); ax = np.atleast_1d(ax)
        for i, c in enumerate(CITIES_OK):
            g = ROLL3[ROLL3["city"] == c]
            for meth in KEY_METHODS:
                gg = g[g["method"] == meth].sort_values("test_year")
                if len(gg): ax[i].plot(gg["test_year"], 100 * gg["max_abs_dev_from_90"], marker="o", label=meth, lw=2.2 if meth.startswith("Mondrian") else 1,
                                       ls="-" if meth.startswith("Mondrian") else ("--" if "ZINB" in meth else ":"))
            ax[i].axhline(3, color="k", ls=":", lw=.8); ax[i].set_title(f"{CITY_NAME[c]}: worst-quintile deviation by test year (points)")
        ax[-1].legend(fontsize=7); savefig("figE2_v3_rolling_with_zinb")

## Part I.5 Benchmark intervals with ZINB baselines (30 validation days as calibration, as in v1)

In [ ]:
with part("I.5 (v3) benchmark intervals with ZINB baselines"):
    BCOV3 = []
    for c in BENCH_CITIES:
        Xall, ntr, nva, nte = bench_load(c); R, Cc, T_, K = Xall.shape; N = R * Cc
        trd, vad, ted = np.arange(30, ntr), np.arange(ntr, ntr + nva), np.arange(ntr + nva, T_)
        (Xtr, ytr), (Xva, yva), (Xte, yte) = (bench_features(Xall, d_) for d_ in (trd, vad, ted))
        vol = pd.Series(Xall[:, :, :ntr].mean(2).reshape(-1)); vq = pd.qcut(vol.rank(method="first"), 5, labels=False).values + 1; vq_rows = np.tile(vq, nte)
        for s in SEEDS_TREE:                                          # our benchmark forecaster (XGBoost-Poisson was selected on validation in both cities)
            mx = fit_xgb_pois(Xtr, ytr, Xva, yva, seed=s); mu, mv = mx.predict(Xte), mx.predict(Xva)
            sc = norm_score(yva, mv); edges = mondrian_edges(mv); bv = np.digitize(mv, edges)
            qb = np.array([conformal_q(sc[bv == i]) for i in range(len(edges) + 1)])
            meths = {"Poisson": parametric_interval(mu, "Poisson"), "Neg. binomial": parametric_interval(mu, "NB", nb_size(mv, yva)),
                     "Split conformal": int_from_q(mu, conformal_q(sc)), "Mondrian conformal": int_from_q(mu, qb[np.digitize(mu, edges)])}
            zres, zpar = zinb_methods(mv, yva, mu); meths.update(zres)
            for meth, (lo, hi) in meths.items():
                r_ = coverage_report(yte, lo, hi, vq_rows); r_["worst_under_coverage"] = max(0.9 - r_[f"cov_Q{q}"] for q in range(1, 6))
                BCOV3.append(dict(city=c, seed=s, method=meth, **r_))
            log(f"  {c} seed {s}: ZINB fitted on the 30 validation days: " + ", ".join(f"{p['variant']} bin {p['bin']}: pi={p['pi']:.2f} r={p['r']:.2f}" for p in zpar[:3]))
    BCOV3 = save_table(pd.DataFrame(BCOV3), "b17_v3_benchmark_intervals_with_zinb")
    display(BCOV3.groupby(["city", "method"])[["coverage", "width", "interval_score", "cov_Q3", "cov_Q4", "cov_Q5", "worst_under_coverage"]].mean().round(3))

## Summary: consistency with v1 and the v3 verdicts
New hypotheses, fixed before this run: **A9** (Mondrian never under-covers by more than 3 points; every ZINB baseline misses by more than 3 points somewhere), **A9b** (Mondrian has the better interval score at H = 1), **A10** (Mondrian-ACI beats every ZINB baseline in 2020), **G1** (all audited count codebases score only non-zero cells).

In [ ]:
def g_(name):
    v = globals().get(name); return v if isinstance(v, pd.DataFrame) and len(v) else None
def verdict(ok, partly=False): return "supported" if ok else ("partly supported" if partly else "not supported")
ZINB_METHODS = ["ZINB (fitted)", "ZINB (fitted per volume bin)", "GRU-ZINB (learned)"]
with part("VI (v3). consistency with v1, new verdicts and summary"):
    # 1. does v3 reproduce v1's interval results? (same method, horizon, city, seed 0)
    d01 = tab_prev("d01_interval_coverage_main_split"); CONS = None
    if d01 is not None and g_("COV3") is not None:
        a = d01[d01["seed"] == d01.groupby(["H", "city"])["seed"].transform("min")][["H", "city", "method", "coverage", "max_abs_dev_from_90"]]
        b = COV3[COV3["seed"] == SEEDS_TREE[0]][["H", "city", "method", "coverage", "max_abs_dev_from_90"]]
        CONS = a.merge(b, on=["H", "city", "method"], suffixes=("_v1", "_v3"))
        CONS["coverage_diff_pts"] = 100 * (CONS["coverage_v3"] - CONS["coverage_v1"]); CONS["worst_dev_diff_pts"] = 100 * (CONS["max_abs_dev_from_90_v3"] - CONS["max_abs_dev_from_90_v1"])
        CONS = save_table(CONS, "d06_v3_consistency_with_v1")
        log(f"  v1 vs v3 (same methods): coverage differs by at most {CONS['coverage_diff_pts'].abs().max():.2f} pts, worst-quintile deviation by at most {CONS['worst_dev_diff_pts'].abs().max():.2f} pts")
    # 2. new hypotheses, stated before this run
    NEW = []
    C_ = g_("COV3")
    if C_ is not None:
        c0 = C_[C_["seed"] == SEEDS_TREE[0]]; mon = c0[c0["method"] == "Mondrian CP"]; ok_m = (mon["worst_under_coverage"] <= 0.03).all()
        zf = {m: c0[c0["method"] == m] for m in ZINB_METHODS if (c0["method"] == m).any()}
        ok_z = all((g["max_abs_dev_from_90"] > 0.03).any() for g in zf.values()) and len(zf) == 3
        NEW.append(dict(hypothesis="A9 (v3)", claim="Mondrian CP keeps worst-quintile under-coverage ≤ 3 points in every city and horizon, while each ZINB baseline (fitted, fitted per volume bin, learned GRU-ZINB) misses 90 % by more than 3 points (either direction) in some volume quintile",
                        verdict=verdict(ok_m and ok_z, ok_m),
                        evidence=f"Mondrian CP worst under-coverage {100 * mon['worst_under_coverage'].max():.1f} pts; " + "; ".join(
                            f"{m}: worst deviation {100 * g['max_abs_dev_from_90'].max():.1f} pts (under {100 * g['worst_under_coverage'].max():.1f}, over {100 * g['worst_over_coverage'].max():.1f})" for m, g in zf.items())))
        h1 = c0[c0["H"] == 1].pivot_table(index="city", columns="method", values="interval_score")
        if {"Mondrian CP"} <= set(h1.columns):
            better = {m: (h1["Mondrian CP"] <= h1[m]).sum() for m in ZINB_METHODS if m in h1.columns}
            NEW.append(dict(hypothesis="A9b (v3)", claim="at H = 1 Mondrian CP has a lower (better) interval score than each ZINB baseline in every city",
                            verdict=verdict(all(v == len(h1) for v in better.values()), any(v > 0 for v in better.values())),
                            evidence="; ".join(f"Mondrian ≤ {m} in {v}/{len(h1)} cities" for m, v in better.items())))
    R_ = g_("ROLL3")
    if R_ is not None:
        r20 = R_[(R_["test_year"] == 2020) & R_["city"].isin(["CHI", "NYC"])].pivot_table(index="city", columns="method", values="max_abs_dev_from_90")
        if "Mondrian-ACI" in r20.columns:
            wins = {m: int((r20["Mondrian-ACI"] < r20[m]).sum()) for m in ZINB_METHODS if m in r20.columns}
            NEW.append(dict(hypothesis="A10 (v3)", claim="in the 2020 fold Mondrian-ACI has a smaller worst-quintile deviation than every ZINB baseline, in Chicago and New York",
                            verdict=verdict(all(v == len(r20) for v in wins.values()) and len(wins) == 3, any(v > 0 for v in wins.values())),
                            evidence="; ".join(f"{c}: Mondrian-ACI {100 * r20.loc[c, 'Mondrian-ACI']:.1f}" + "".join(f", {m} {100 * r20.loc[c, m]:.1f}" for m in wins) + " pts" for c in r20.index)))
    W_ = g_("WIDE")
    if W_ is not None:
        reg = W_[W_["task"] == "regression"]; n_m = int(reg["test metric ignores zero cells"].sum())
        NEW.append(dict(hypothesis="G1 (v3)", claim="every audited public crime-count (regression) codebase scores only non-zero cells in its test metric",
                        verdict=verdict(n_m == len(reg) and len(reg) > 0, n_m > 0),
                        evidence=f"{n_m}/{len(reg)} regression codebases ({', '.join(reg[reg['test metric ignores zero cells']]['code'])}); "
                                 f"direct evidence for {int((reg['evidence'].str.startswith('direct') & reg['test metric ignores zero cells']).sum())}, indirect for the rest"))
    B_ = g_("BCOV3")                                               # benchmark: reported, not tested (cells with no crime make every method cover Q1–Q2 fully)
    NEW = pd.DataFrame(NEW); HYP2 = tab_prev("z01_hypothesis_verdicts")
    HYP = save_table(pd.concat([HYP2, NEW], ignore_index=True) if HYP2 is not None else NEW, "z01_hypothesis_verdicts"); display(HYP)

    # 3. summary file: the v2 summary plus a v3 section
    def md(df, n=80):
        try: return df.head(n).to_markdown(index=False, floatfmt=".4f")
        except Exception: return df.head(n).to_string(index=False)
    lines = [f"# Option A experiments – results summary (v1 + v2 + v3)", f"v3 generated {pd.Timestamp.now():%Y-%m-%d %H:%M}, run time {hours_used():.2f} h, FAST = {FAST}.",
             "v3 adds: a protocol census of public crime-prediction code (g01–g02), ZINB interval baselines on the weekly panels (d03–d05, e03) and on the benchmark (b17), "
             "and a consistency check of the re-built panels against v1 (c02b, d06). Everything else is from v1/v2 (see results_summary_v2.md).", "",
             "## v3 part status", md(pd.DataFrame([dict(part=k, status=v) for k, v in STATUS.items()])), "", "## All hypotheses (v1–v3)", md(HYP), ""]
    if W_ is not None: lines += ["## Protocol census of public crime-prediction code", md(W_), ""]
    if C_ is not None:
        k = C_[(C_["seed"] == SEEDS_TREE[0]) & (C_["H"] == 1) & C_["method"].isin(KEY_METHODS + ["CQR", "Locally weighted CP", "ACI"])]
        lines += ["## Weekly intervals with ZINB baselines (H = 1, seed 0)", md(k[["city", "method", "coverage", "width", "interval_score", "cov_Q1", "cov_Q2", "cov_Q3", "cov_Q4", "cov_Q5",
                                                                             "worst_under_coverage", "worst_over_coverage"]], 200), ""]
        lines += ["## Worst-quintile under-coverage, all horizons (seed 0, points)", md((C_[C_["seed"] == SEEDS_TREE[0]].pivot_table(index="method", columns=["city", "H"], values="worst_under_coverage") * 100).round(1).reset_index()), ""]
    if R_ is not None: lines += ["## Rolling origins with ZINB: worst-quintile deviation (points)", md((R_.pivot_table(index="method", columns=["city", "test_year"], values="max_abs_dev_from_90") * 100).round(1).reset_index()), ""]
    if B_ is not None: lines += ["## Benchmark intervals with ZINB (mean over seeds)", md(B_.groupby(["city", "method"])[["coverage", "width", "interval_score", "worst_under_coverage"]].mean().reset_index()), ""]
    if g_("ZPAR") is not None: lines += ["## Fitted ZINB parameters (H = 1, seed 0)", md(ZPAR[(ZPAR["H"] == 1) & (ZPAR["seed"] == SEEDS_TREE[0])]), ""]
    if CONS is not None: lines += ["## Consistency with v1 (same methods)", md(CONS.groupby("method")[["coverage_diff_pts", "worst_dev_diff_pts"]].agg(lambda s: s.abs().max()).reset_index()), ""]
    if g_("CHK") is not None: lines += ["## Panels re-downloaded in v3 vs v1", md(CHK), ""]
    prev_md = (PREV / "results_summary.md").read_text() if (PREV / "results_summary.md").exists() else ""
    (OUT / "results_summary.md").write_text("\n".join(lines) + "\n\n---\n\n" + prev_md)
    json.dump(dict(status=STATUS, status_prev=STATUS_PREV, hours=hours_used(), versions=VERSIONS, hypotheses=HYP.to_dict("records")), open(OUT / "metrics_summary.json", "w"), indent=1, default=str)
    log("results_summary.md written")
print(f"Finished in {hours_used():.2f} h. Download /kaggle/working/optionA_outputs_v3.zip from the Output tab.")